# Глубинное обучение 1 / Введение в глубинное обучение, ФКН ВШЭ

## Домашнее задание 2: сверточный классификатор.

### Общая информация

Оценка после штрафа после мягкого дедлайна вычисляется по формуле $M_{\text{penalty}} = M_{\text{full}} \cdot 0.85^{t/1440}$, где $M_{\text{full}}$ — полная оценка за работу без учета штрафа, а $t$ — время в минутах, прошедшее после мягкого дедлайна (округление до двух цифр после запятой). Таким образом, спустя первые сутки после мягкого дедлайна вы не можете получить оценку выше 8.5, а если сдать через четыре дня после мягкого дедлайна, то ваш максимум — 5.22 балла.

### Оценивание и штрафы

Максимально допустимая оценка за работу — 10 баллов (+бонусы). Сдавать задание после указанного срока сдачи нельзя.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

### Правила экспериментов

- В ячейке ниже вычисляется `STUDENT_ID`, не забудьте изменить значение по умолчанию в ней! При запуске обучения используйте сиды `STUDENT_ID`, `STUDENT_ID + 1`, `STUDENT_ID + 2`.
- Ответы должны интерпретировать *ваши* графики и таблицы. Ответы, в которых нет ссылок на ваши числа, оцениваются в 0 баллов. Помимо кода, оцениваются оформление графиков (подписи осей, легенда) и таблиц.

### Технические требования

- Для выполнения задания нужна GPU (достаточно Colab). Ориентировочный суммарный бюджет вычислений — 2–3 часа GPU, планируйте эксперименты заранее.
- Если эксперименты не помещаются в бюджет, увеличьте `batch_size` (128 и больше, при необходимости пересмотрев `lr`), включите `pin_memory` и `persistent_workers` в `DataLoader`, при желании используйте смешанную точность и храните датасет на GPU.

### О задании

В этом задании вам предстоит познакомиться со сверточными сетями и их обучением для классификации изображений с использованием библиотеки PyTorch.

### Как запускать решение

Код ниже заполняет задания 1–7 и бонус с TensorBoard. Обучение пока не запускалось: чисел и готовых выводов о качестве здесь нет.

1. Укажите фамилию в `STUDENT_SURNAME`. Один и тот же train/val split используется во всех опытах; меняются только три сида обучения.
2. В Colab выберите GPU. При необходимости подключите Drive и задайте `RESULTS_DIR` на нём: файлы в `/content` исчезают после удаления сессии.
3. По умолчанию `RUN_TRAINING = False`, `RUN_DATA = False`, а `STAGES` пуст. Поэтому «Выполнить всё» не скачивает CIFAR-10 и не начинает обучение.
4. Для экспериментов включите `RUN_DATA`, `RUN_TRAINING` и добавляйте разделы в `STAGES` по порядку: `"2"`, `"3"`, `"4"`, `"5"`, затем `"7"`. Код использует готовые результаты предыдущих разделов из `RESULTS_DIR`. Каждый опыт сохраняется отдельно; завершённые запуски повторно не обучаются. Прерванный запуск одного сида начинается заново.
5. После выбора моделей включите `RUN_REPORTS` и раздел `"6"`. Для финальной оценки челленджа отдельно включите `FINAL_TEST`. Выбор настроек не обращается к test.

В таблицах используются три запуска и выборочное стандартное отклонение (`ddof=1`). Правило «больше одного std» применяю консервативно: разница средних должна превышать большее из двух std. Это правило задания, а не статистический тест. Текст под таблицами строится из результатов; без запусков он сообщает об отсутствии данных.

Если меняете реализацию сети, обучения или преобразований, увеличьте `SCHEMA_VERSION` в начальной ячейке: тогда код не возьмёт результаты прежней реализации из кэша.


In [ ]:
STUDENT_SURNAME = "vorsin"  # Написание фамилии подтверждено.
STUDENT_ID = sum(map(ord, STUDENT_SURNAME)) if STUDENT_SURNAME.strip() else None
SEEDS = [STUDENT_ID + i for i in range(3)] if STUDENT_ID is not None else []
print("STUDENT_ID:", STUDENT_ID, "seeds:", SEEDS)


In [ ]:
import copy
import hashlib
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision.transforms import v2
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)
classes = (
    "plane",
    "car",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
)
n_classes = len(classes)

RUN_DATA = False
RUN_TRAINING = False
RUN_REPORTS = False
RUN_BENCHMARK = False
FINAL_TEST = False
STAGES = set()  # Например, {"2"}; добавляйте следующие разделы после предыдущих.
RESULTS_DIR = Path("./hw2-results")  # В Colab лучше указать путь на Google Drive.
DATA_DIR = Path("./data")
BATCH_SIZE = 128
LOADER_OPTIONS = dict(num_workers=2, pin_memory=True, persistent_workers=False)
AMP = True
AUG_EPOCHS = 20  # Разрешено уменьшить до 10, одинаково для всех строк раздела 5.
USE_TENSORBOARD = False
SCHEMA_VERSION = 1


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def stage_enabled(stage):
    return RUN_TRAINING and stage in STAGES


def need_identity():
    if STUDENT_ID is None:
        raise ValueError("Сначала заполните STUDENT_SURNAME.")


print(f"torch={torch.__version__}, torchvision={torchvision.__version__}")
print("Обучение разрешено:", RUN_TRAINING, "; разделы:", STAGES)


## 0. Загрузка данных

Работать мы будем с набором данных [CIFAR10](https://www.cs.toronto.edu/~kriz/cifar.html). CIFAR10 представляет собой набор изображений 32х32 пикселя, разделенных на 10 классов.

![title](https://pytorch.org/tutorials/_images/cifar10.png)




Набор данных уже определен в <code>torchvision.datasets</code>, так что возьмем его оттуда.

Функция `get_cifar10_data` ниже использует фиксированный сид `0`. Замените его на значение, зависящее от вашего `STUDENT_ID` (см. правила в начале ноутбука).

In [ ]:
def make_transform(name="none", erase_p=0.25):
    ops = [v2.ToImage()]
    if name != "none":
        if name != "flip":
            ops.append(v2.RandomCrop(32, padding=4))
        ops.append(v2.RandomHorizontalFlip())
    if name == "trivial":
        ops.append(v2.TrivialAugmentWide())
    elif name == "rand":
        ops.append(v2.RandAugment(num_ops=2, magnitude=9))
    elif name not in {"none", "flip", "base", "erase", "mixup", "cutmix", "rand_erase"}:
        raise ValueError(name)
    if name == "rand_erase":
        ops.append(v2.RandAugment(num_ops=2, magnitude=9))
    ops.extend(
        [v2.ToDtype(torch.float32, scale=True), v2.Normalize(CIFAR10_MEAN, CIFAR10_STD)]
    )
    if name in {"erase", "rand_erase"}:
        ops.append(
            v2.RandomErasing(p=erase_p, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0)
        )
    return v2.Compose(ops)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_cifar10_data(
    batch_size,
    transform_train,
    transform_eval=None,
    valid_size=0.3,
    num_workers=2,
    seed=None,
    pin_memory=True,
    persistent_workers=False,
):
    need_identity()
    if not RUN_DATA:
        raise RuntimeError("Загрузка данных выключена: RUN_DATA = False.")
    seed = STUDENT_ID if seed is None else seed
    seed_everything(seed)
    transform_eval = (
        make_transform("none") if transform_eval is None else transform_eval
    )
    train_full = torchvision.datasets.CIFAR10(
        DATA_DIR, train=True, download=True, transform=transform_train
    )
    eval_full = torchvision.datasets.CIFAR10(
        DATA_DIR, train=True, download=True, transform=transform_eval
    )
    indices = np.arange(len(train_full))
    train_idx, valid_idx = train_test_split(
        indices, test_size=valid_size, shuffle=True, random_state=STUDENT_ID
    )
    common = dict(
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers and num_workers > 0,
        worker_init_fn=seed_worker,
    )
    train_loader = DataLoader(
        Subset(train_full, train_idx),
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        **common
    )
    val_loader = DataLoader(Subset(eval_full, valid_idx), shuffle=False, **common)
    clean_train_loader = DataLoader(
        Subset(eval_full, train_idx), shuffle=False, **common
    )
    # Test не скачивается и не читается до разделов итоговой оценки.
    return train_loader, val_loader, clean_train_loader


def get_test_loader():
    if not (RUN_REPORTS or FINAL_TEST) or not RUN_DATA:
        raise RuntimeError("Для test нужны RUN_DATA и RUN_REPORTS либо FINAL_TEST.")
    testset = torchvision.datasets.CIFAR10(
        DATA_DIR, train=False, download=True, transform=make_transform("none")
    )
    return DataLoader(
        testset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=LOADER_OPTIONS["num_workers"],
        pin_memory=LOADER_OPTIONS["pin_memory"],
    )


In [ ]:
train_loader = val_loader = clean_train_loader = None
if RUN_DATA:
    train_loader, val_loader, clean_train_loader = get_cifar10_data(
        BATCH_SIZE, make_transform("none"), **LOADER_OPTIONS
    )
else:
    print("Данные не загружались. Включите RUN_DATA, когда будете готовы.")


Посмотрим на изображения:

In [ ]:
def denormalize(images):
    mean = images.new_tensor(CIFAR10_MEAN).view(-1, 1, 1)
    std = images.new_tensor(CIFAR10_STD).view(-1, 1, 1)
    return (images * std + mean).clamp(0, 1)


def imshow(img, title=None):
    plt.figure(figsize=(8, 3))
    plt.imshow(denormalize(img.detach().cpu()).permute(1, 2, 0))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


if train_loader is not None:
    images, labels = next(iter(train_loader))
    imshow(
        torchvision.utils.make_grid(images[:4]),
        ", ".join(classes[int(y)] for y in labels[:4]),
    )


## 1. Задание сверточной сети (0.75 балла)

Теперь нам нужно задать сверточную нейронную сеть, которую мы будем обучать классифицировать изображения.

Используем сеть, основанную на одном блоке архитектуры ResNet.

<img src="https://i.ibb.co/2hg962h/basic-block.png" width="500"/>

__Указания:__

- Все сверточные слои должны иметь 32 выходных канала, а также не должны изменять ширину и высоту изображения.
- Выход блока сократите до размерности 32х4х4, применив average pooling.
- Для получения итоговых логитов, распрямите выход пулинга в вектор из 512 элементов, а затем пропустите его через линейный слой.

**Задание 1.1 (0.75 балла).**

Определите архитектуру сети соответственно схеме и указаниям выше.

Ключевые слова: <code>Conv2d</code>, <code>BatchNorm2d</code>, <code>AvgPool2d</code>.

In [ ]:
class BasicBlockNet(nn.Module):
    def __init__(self, skip=True, bn=True):
        super().__init__()
        self.skip = skip
        # Bias оставлен и при BN: это позволяет менять только сам BN при абляции.
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1, bias=True)
        self.bn1 = nn.BatchNorm2d(32) if bn else nn.Identity()
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1, bias=True)
        self.bn2 = nn.BatchNorm2d(32) if bn else nn.Identity()
        self.shortcut = nn.Conv2d(3, 32, 1, bias=True) if skip else None
        self.pool = nn.AvgPool2d(8)
        self.fc = nn.Linear(32 * 4 * 4, n_classes)

    def features(self, x):
        out = self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))
        if self.skip:
            out = out + self.shortcut(x)
        return F.relu(out)

    def forward(self, x):
        return self.fc(self.pool(self.features(x)).flatten(1))


На схеме skip-ветка содержит свёртку 1×1. Она переводит RGB-вход из 3 каналов в 32, поэтому его можно сложить с основной веткой. Внутри блока две свёртки 3×3 с `padding=1`. После пулинга получается `(N, 32, 4, 4)`, затем `(N, 512)` и десять логитов. Softmax перед `cross_entropy` не нужен: функция сама вычисляет логарифм нормированных вероятностей.


In [ ]:
net = BasicBlockNet()
net


Проверим, что выход сети имеет корректную размерность:

In [ ]:
with torch.no_grad():
    assert net(torch.zeros(2, 3, 32, 32)).shape == (2, 10)


Чтобы проводить вычисления на GPU, в PyTorch необходимо руками перекладывать объекты, с которыми вы хотите проводить вычисления, на графический ускоритель. Это делается следующим образрм:

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
# Перенос на GPU выполняет run_experiment непосредственно перед обучением.
net = net.cpu()


Подключение GPU в google.colab:

**Среда выполнения** -> **Сменить среду выполнения** -> **Аппаратный ускоритель** -> **GPU**

## 2. Обучение и тестирование модели (1 балл)

Пара важных замечаний:
- Все сравнения и выбор гиперпараметров делаются по train/val. Тестовая выборка используется только для итоговой отчетности (разделы 6 и 7), но не для выбора модели.
- Здесь и далее любое утверждение вида «стало лучше/хуже» **должно опираться на 3 запуска** с сидами `STUDENT_ID + ...` и сравнение mean ± std. Разница меньше одного std не считается улучшением.

**Задание 2.1 (0.5 балла).** Переходим к обучению модели. Заполните пропуски в функциях `test` и `train_epoch`. В качестве функции потерь будем использовать [кросс-энтропию](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html) (с учетом параметра `label_smoothing`, который понадобится в разделе 3), а в качестве метрики качества accuracy.

In [ ]:
def soft_cross_entropy(logits, targets, label_smoothing=0.0):
    targets = targets * (1 - label_smoothing) + label_smoothing / logits.shape[1]
    return -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()


def classification_loss(logits, targets, label_smoothing=0.0):
    if targets.ndim == 2:
        return soft_cross_entropy(logits, targets, label_smoothing)
    return F.cross_entropy(logits, targets, label_smoothing=label_smoothing)


def batch_accuracy(logits, targets):
    pred = logits.argmax(1)
    if targets.ndim == 2:
        # Масса мягкой метки, которую получил предсказанный класс; не обычная accuracy.
        return targets.gather(1, pred[:, None]).mean()
    return (pred == targets).float().mean()


@torch.no_grad()
def test(model, loader, label_smoothing=0.0):
    model.eval()
    model_device = next(model.parameters()).device
    loss_sum = correct_sum = n = 0
    for data, target in loader:
        data = data.to(model_device, non_blocking=True)
        target = target.to(model_device, non_blocking=True)
        logits = model(data)
        batch_n = target.shape[0]
        loss_sum += (
            classification_loss(logits, target, label_smoothing).item() * batch_n
        )
        correct_sum += (logits.argmax(1) == target).sum().item()
        n += batch_n
    if n == 0:
        raise ValueError("Пустой загрузчик данных.")
    if not math.isfinite(loss_sum):
        raise FloatingPointError("Validation loss перестал быть конечным.")
    return loss_sum / n, correct_sum / n


def train_epoch(
    model,
    optimizer,
    train_loader,
    label_smoothing=0.0,
    scheduler=None,
    scheduler_per_batch=False,
    mixer=None,
    scaler=None,
):
    model.train()
    model_device = next(model.parameters()).device
    use_amp = AMP and model_device.type == "cuda"
    loss_log, acc_log, lr_log, sizes = [], [], [], []
    for data, target in train_loader:
        data = data.to(model_device, non_blocking=True)
        target = target.to(model_device, non_blocking=True)
        if mixer is not None:
            data, target = mixer(data, target)
        optimizer.zero_grad(set_to_none=True)
        lr_log.append(optimizer.param_groups[0]["lr"])
        with torch.autocast(device_type=model_device.type, enabled=use_amp):
            logits = model(data)
            loss = classification_loss(logits, target, label_smoothing)
        if not torch.isfinite(loss):
            raise FloatingPointError("Loss перестал быть конечным.")
        if scaler is not None and use_amp:
            old_scale = scaler.get_scale()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            stepped = scaler.get_scale() >= old_scale
        else:
            loss.backward()
            optimizer.step()
            stepped = True
        if scheduler is not None and scheduler_per_batch and stepped:
            scheduler.step()
        loss_log.append(loss.item())
        acc_log.append(batch_accuracy(logits.detach(), target).item())
        sizes.append(len(data))
    return loss_log, acc_log, lr_log, sizes


def train(
    model,
    optimizer,
    n_epochs,
    train_loader,
    val_loader,
    scheduler=None,
    label_smoothing=0.0,
    weight_decay=0.0,
    scheduler_per_batch=False,
    mixer=None,
    checkpoint_callback=None,
    writer=None,
):
    # Явный аргумент, одинаково применяемый ко всем группам параметров.
    for group in optimizer.param_groups:
        group["weight_decay"] = weight_decay
    scaler = torch.amp.GradScaler(
        "cuda", enabled=AMP and next(model.parameters()).is_cuda
    )
    history = dict(
        train_loss=[],
        train_acc=[],
        lr=[],
        val_loss=[],
        val_acc=[],
        epoch_train_loss=[],
        epoch_train_acc=[],
        epoch_seconds=[],
        val_steps=[],
    )
    best_acc, best_state, best_epoch = -1, None, None
    for epoch in range(n_epochs):
        if next(model.parameters()).is_cuda:
            torch.cuda.synchronize()
        started = time.perf_counter()
        losses, accs, lrs, sizes = train_epoch(
            model,
            optimizer,
            train_loader,
            label_smoothing,
            scheduler,
            scheduler_per_batch,
            mixer,
            scaler,
        )
        if next(model.parameters()).is_cuda:
            torch.cuda.synchronize()
        elapsed = (
            time.perf_counter() - started
        )  # Только обучение, без val и записи весов.
        val_loss, val_acc = test(model, val_loader, label_smoothing=0.0)
        history["train_loss"].extend(losses)
        history["train_acc"].extend(accs)
        history["lr"].extend(lrs)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_steps"].append(len(history["train_loss"]))
        history["epoch_train_loss"].append(float(np.average(losses, weights=sizes)))
        history["epoch_train_acc"].append(float(np.average(accs, weights=sizes)))
        history["epoch_seconds"].append(elapsed)
        if val_acc > best_acc:
            best_acc, best_epoch = val_acc, epoch + 1
            best_state = {
                k: v.detach().cpu().clone() for k, v in model.state_dict().items()
            }
        if scheduler is not None and not scheduler_per_batch:
            scheduler.step()
        if writer is not None:
            offset = len(history["train_loss"]) - len(losses)
            for j, (loss, acc, lr) in enumerate(
                zip(losses, accs, lrs), start=offset + 1
            ):
                writer.add_scalar("loss/train", loss, j)
                writer.add_scalar("accuracy/train_augmented", acc, j)
                writer.add_scalar("lr", lr, j)
            writer.add_scalar("loss/val", val_loss, len(history["train_loss"]))
            writer.add_scalar("accuracy/val", val_acc, len(history["train_loss"]))
        if checkpoint_callback:
            checkpoint_callback(history, best_state, best_epoch)
        print(
            f"{epoch + 1:02d}/{n_epochs}: val acc={val_acc:.4f}, CE={val_loss:.4f}, train={elapsed:.1f}s"
        )
    return history, best_state, best_epoch


@dataclass(frozen=True)
class Config:
    model: str = "basic"
    optimizer: str = "sgd"
    lr: float = 0.1
    schedule: str = "constant"
    weight_decay: float = 0.0
    label_smoothing: float = 0.0
    augmentation: str = "none"
    mixing: str = "none"
    epochs: int = 20
    batch_size: int = BATCH_SIZE
    width: int = 32
    num_workers: int = 2
    pin_memory: bool = True
    persistent_workers: bool = False


def build_model(config):
    if config.model in {"basic", "no_skip", "no_bn", "neither"}:
        # Инициализируем полную сеть, потом убираем части: общие веса строго одинаковы.
        model = BasicBlockNet()
        if config.model in {"no_skip", "neither"}:
            model.skip, model.shortcut = False, None
        if config.model in {"no_bn", "neither"}:
            model.bn1 = model.bn2 = nn.Identity()
        return model
    if config.model == "resnet":
        return SmallResNet(config.width)
    if config.model == "convnext":
        return SmallConvNeXt(config.width)
    if config.model == "challenge":
        return ChallengeResNet(config.width)
    raise ValueError(config.model)


def make_optimizer(model, config):
    kwargs = dict(lr=config.lr, weight_decay=config.weight_decay)
    if config.optimizer == "sgd":
        return optim.SGD(model.parameters(), momentum=0.9, **kwargs)
    if config.optimizer == "adamw":
        return optim.AdamW(model.parameters(), **kwargs)
    raise ValueError(config.optimizer)


def make_scheduler(optimizer, config, steps_per_epoch):
    if config.schedule == "constant":
        return None, False
    if config.schedule == "multistep":
        return (
            optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10, 15], gamma=0.1),
            False,
        )
    if config.schedule == "onecycle":
        return (
            optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=config.lr,
                total_steps=config.epochs * steps_per_epoch,
                pct_start=0.3,
                cycle_momentum=False,
            ),
            True,
        )
    if config.schedule == "cosine":
        warmup = min(3, config.epochs - 1)
        return (
            optim.lr_scheduler.SequentialLR(
                optimizer,
                schedulers=[
                    optim.lr_scheduler.LinearLR(
                        optimizer, start_factor=0.1, total_iters=warmup
                    ),
                    optim.lr_scheduler.CosineAnnealingLR(
                        optimizer,
                        T_max=config.epochs - warmup,
                        eta_min=config.lr * 0.001,
                    ),
                ],
                milestones=[warmup],
            ),
            False,
        )
    raise ValueError(config.schedule)


def make_mixer(name):
    if name == "none":
        return None
    if name == "mixup":
        return v2.MixUp(alpha=0.2, num_classes=n_classes)
    if name == "cutmix":
        return v2.CutMix(alpha=1.0, num_classes=n_classes)
    raise ValueError(name)


def experiment_path(config, seed):
    payload = dict(
        config=asdict(config),
        seed=seed,
        student=STUDENT_ID,
        schema=SCHEMA_VERSION,
        amp=AMP,
        torch=torch.__version__,
        torchvision=torchvision.__version__,
    )
    key = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]
    return RESULTS_DIR / key


def run_experiment(name, config, seed, stage):
    path = experiment_path(config, seed)
    result_file = path / "result.json"
    if result_file.exists():
        cached = json.loads(result_file.read_text())
        cached["path"] = str(path)
        return cached
    if not stage_enabled(stage):
        return None
    need_identity()
    if not torch.cuda.is_available():
        raise RuntimeError("Полные опыты рассчитаны на GPU; выберите GPU в Colab.")
    path.mkdir(parents=True, exist_ok=True)
    seed_everything(seed)
    loaders = get_cifar10_data(
        config.batch_size,
        make_transform(config.augmentation),
        seed=seed,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        persistent_workers=config.persistent_workers,
    )
    tr, val, clean = loaders
    # Создание DataLoader тоже меняет RNG: явно фиксируем начало инициализации.
    seed_everything(seed)
    model = build_model(config).to(device)
    optimizer = make_optimizer(model, config)
    scheduler, per_batch = make_scheduler(optimizer, config, len(tr))
    writer = None
    if USE_TENSORBOARD:
        from torch.utils.tensorboard import SummaryWriter

        writer = SummaryWriter(str(path / "tensorboard"))

    def save_progress(history, state, epoch):
        (path / "partial-history.json").write_text(json.dumps(history))
        torch.save(state, path / "best.pt")

    try:
        history, state, epoch = train(
            model,
            optimizer,
            config.epochs,
            tr,
            val,
            scheduler,
            label_smoothing=config.label_smoothing,
            weight_decay=config.weight_decay,
            scheduler_per_batch=per_batch,
            mixer=make_mixer(config.mixing),
            checkpoint_callback=save_progress,
            writer=writer,
        )
        # Разрыв train−val считаем на чистых изображениях у одной и той же модели.
        clean_last_loss, clean_last_acc = test(model, clean)
        last_val_loss, last_val_acc = test(model, val)
        model.load_state_dict(state)
        selected_loss, selected_acc = test(model, val)
        result = dict(
            name=name,
            config=asdict(config),
            seed=seed,
            path=str(path),
            val_acc=selected_acc,
            val_loss=selected_loss,
            best_epoch=epoch,
            train_acc=clean_last_acc,
            train_loss=clean_last_loss,
            last_val_acc=last_val_acc,
            last_val_loss=last_val_loss,
            gap=clean_last_acc - last_val_acc,
            train_aug_acc=history["epoch_train_acc"][-1],
            epoch_seconds=float(np.mean(history["epoch_seconds"])),
            history=history,
            status="complete",
        )
    except FloatingPointError as error:
        result = dict(
            name=name,
            config=asdict(config),
            seed=seed,
            path=str(path),
            status="diverged",
            error=str(error),
        )
    finally:
        if writer is not None:
            writer.close()
        del model, optimizer, loaders, tr, val, clean
        torch.cuda.empty_cache()
    temp = path / "result.tmp"
    temp.write_text(json.dumps(result, ensure_ascii=False, indent=2))
    temp.replace(result_file)
    return result


RESULTS = {}


def group(name, config, stage, seeds=None):
    seeds = SEEDS if seeds is None else seeds
    rows = [run_experiment(name, config, s, stage) for s in seeds]
    rows = [r for r in rows if r is not None]
    RESULTS[name] = rows
    return rows


def complete(rows):
    return len(rows) == 3 and all(r["status"] == "complete" for r in rows)


def stats(rows, metric="val_acc"):
    values = np.array([r[metric] for r in rows], dtype=float)
    return float(values.mean()), float(values.std(ddof=1))


def fmt(rows, metric="val_acc", percent=False):
    if not complete(rows):
        return f"завершено {sum(r['status']=='complete' for r in rows)}/3; расходится {sum(r['status']=='diverged' for r in rows)}"
    mean, std = stats(rows, metric)
    scale = 100 if percent else 1
    return f"{mean*scale:.3f} ± {std*scale:.3f}"


def summary(names, metrics=("val_acc", "val_loss", "epoch_seconds")):
    table = pd.DataFrame(
        [
            dict(Опыт=name, **{m: fmt(RESULTS.get(name, []), m) for m in metrics})
            for name in names
        ]
    )
    display(table)
    return table


def attempted(rows):
    return len(rows) == 3 and all(r["status"] in {"complete", "diverged"} for r in rows)


def best_config(names):
    eligible = [name for name in names if complete(RESULTS.get(name, []))]
    if (
        not names
        or not all(attempted(RESULTS.get(name, [])) for name in names)
        or not eligible
    ):
        return None
    winner = max(eligible, key=lambda name: stats(RESULTS[name])[0])
    return Config(**RESULTS[winner][0]["config"])


def compare(a, b):
    ra, rb = RESULTS.get(a, []), RESULTS.get(b, [])
    if not (complete(ra) and complete(rb)):
        display(
            Markdown(f"Для сравнения `{a}` и `{b}` нужны по три завершённых запуска.")
        )
        return
    ma, sa = stats(ra)
    mb, sb = stats(rb)
    delta = mb - ma
    verdict = "превышает" if abs(delta) > max(sa, sb) else "не превышает"
    display(
        Markdown(
            f"`{a}`: {100*ma:.2f} ± {100*sa:.2f}%; `{b}`: "
            f"{100*mb:.2f} ± {100*sb:.2f}%. Изменение второго относительно первого: "
            f"{100*delta:+.2f} п.п.; модуль разницы {verdict} один std."
            " Это сравнение по val, без обращения к test."
        )
    )


def plot_history(rows, title):
    if not complete(rows):
        print(title, ": нужны 3 успешных запуска.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for i, row in enumerate(rows):
        h = row["history"]
        label = f"seed {row['seed']}"
        for ax, metric in zip(axes, ["loss", "acc"]):
            ax.plot(
                np.arange(1, len(h["train_" + metric]) + 1),
                h["train_" + metric],
                alpha=0.3,
                linewidth=0.6,
                label="train " + label,
            )
            ax.plot(
                h["val_steps"], h["val_" + metric], marker=".", label="val " + label
            )
            ax.set_xlabel("Шаг оптимизатора")
            ax.set_ylabel("Loss" if metric == "loss" else "Accuracy")
            ax.grid(alpha=0.2)
            ax.legend(fontsize=7)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


Запустим обучение модели. В качестве оптимизатора для базового запуска будем использовать стохастический градиентный спуск с моментом.

__Замечание:__ Для достижения наилучшего качества в нашем случае потребуется обучать модель несколько сотен эпох. Однако в целях экономии вашего времени и сил, пока мы ограничимся 20 эпохами (если в условии не сказано иное).

In [ ]:
baseline_config = Config()
baseline_rows = group("baseline", baseline_config, "2")


Посчитайте метрики на валидационной выборке:

In [ ]:
summary(["baseline"])
if complete(baseline_rows):
    print("Средняя val accuracy ≥ 0.67:", stats(baseline_rows)[0] >= 0.67)


Для базовой реализации ожидаем validation accuracy $\geqslant0.67$.

**Задание 2.2 (0.5 балла).** Постройте графики зависимости функции потерь и точности классификации от номера шага оптимизатора. На каждом графике расположите данные и для обучающей и для валидационной выборки, итого у вас должно получиться два графика. Обратите внимание, что на обучающей выборке эти данные считаются по каждому батчу, на валидационной же они считаются по всей выборке раз в эпоху.

In [ ]:
plot_history(baseline_rows, "Базовая сеть, постоянный lr=0.1")


## 3. Оптимизаторы и расписание длины шага (1.5 балла)

С курса "Машинное обучение 1" вы уже должны знать, что сходимость стохастического градиентного спуска мы можем теоретически гарантировать только если будем определенным образом со временем уменьшать длину шага. На практике при обучении нейронных сетей такая техника оказывается очень полезной, однако теоретически обоснованными способами уменьшения длины шага фантазия не ограничивается.

Одним из простейших способов является кусочно постоянная функция: на нескольких фиксированных эпохах уменьшаем длину шага в константу раз. Современные рецепты чаще используют плавные расписания с разогревом (warmup) в начале обучения.

In [ ]:
multistep_config = replace(baseline_config, schedule="multistep")
multistep_rows = group("multistep", multistep_config, "3")


Посчитайте метрики на валидационной выборке для этой конфигурации:

In [ ]:
summary(["multistep"])


**Задание 3.1 (0.25 балла).** Здесь может возникнуть вопрос: а что будет, если мы не будем уменьшать длину шага в процессе обучения, а сразу возьмем констатную, равную значению нашей кусочно-постоянной функции на последних эпохах, то есть $0.001$ в нашем случае. Запустите обучение и сравните с `MultiStepLR` по val. Получилось лучше / хуже?

In [ ]:
small_lr_rows = group("constant_0.001", replace(baseline_config, lr=0.001), "3")
compare("multistep", "constant_0.001")


**Задание 3.2 (0.5 балла).** Изучите, какие способы изменения длины шага представлены в `torch.optim.lr_scheduler`. Обязательно рассмотрите:

- `OneCycleLR`;
- косинусное убывание (`CosineAnnealingLR`) с линейным разогревом в первые эпохи (разогрев и основную часть можно объединить, например, через `SequentialLR`).

Для каждого расписания объясните, как оно устроено, постройте график длины шага от номера шага оптимизатора и сравните качество на val с `MultiStepLR`. Обратите внимание, что разные расписания рассчитаны на вызов `scheduler.step()` с разной частотой (раз в эпоху или раз в батч); при необходимости измените функцию `train`.

In [ ]:
schedule_names = ["multistep", "onecycle", "cosine"]
for schedule in ["onecycle", "cosine"]:
    group(schedule, replace(baseline_config, schedule=schedule), "3")
summary(schedule_names)
for name in ["onecycle", "cosine"]:
    compare("multistep", name)


def plot_schedules(steps_per_epoch=100):
    # Только числа расписания: веса сети не обучаются.
    fig, ax = plt.subplots(figsize=(9, 3))
    for schedule in ["multistep", "onecycle", "cosine"]:
        c = replace(baseline_config, schedule=schedule)
        parameter = nn.Parameter(torch.zeros(()))
        optimizer = optim.SGD([parameter], lr=c.lr)
        scheduler, per_batch = make_scheduler(optimizer, c, steps_per_epoch)
        lrs = []
        for epoch in range(c.epochs):
            for _ in range(steps_per_epoch):
                lrs.append(optimizer.param_groups[0]["lr"])
                optimizer.step()
                if per_batch:
                    scheduler.step()
            if not per_batch:
                scheduler.step()
        ax.plot(np.arange(len(lrs)), lrs, label=schedule)
    ax.set(xlabel="Шаг оптимизатора", ylabel="Learning rate")
    ax.legend()
    ax.grid(alpha=0.2)
    plt.show()


plot_schedules(len(train_loader) if train_loader is not None else 100)
schedule_recipe = best_config(["onecycle", "cosine"])


`MultiStepLR` снижает шаг в 10 раз после 10-й и 15-й эпох. `OneCycleLR` сначала увеличивает его до `max_lr`, затем уменьшает; здесь он вызывается после каждого обновления весов. У cosine-рецепта первые три эпохи шаг растёт линейно от 0.1×lr до lr, затем убывает по косинусу. Для него `step()` вызывается раз в эпоху. На графике без данных показаны условные 100 батчей на эпоху; при загрузке CIFAR используется настоящее число батчей.


**Задание 3.3 (0.75 балла).** Сравните оптимизаторы `SGD` (с моментом) и `AdamW` и исследуйте влияние `weight_decay` и `label_smoothing`. Используйте лучшее расписание из 3.2 для обоих оптимизаторов.

1. Подберите по val `lr` для каждого оптимизатора на небольшой сетке (например, для `SGD` $\{0.03, 0.1, 0.3\}$, для `AdamW` $\{3 \cdot 10^{-4}, 10^{-3}, 3 \cdot 10^{-3}\}$; на этом шаге достаточно одного запуска на значение).
2. С лучшими `lr` сравните `SGD` и `AdamW` (3 запуска, mean ± std).
3. Для лучшего оптимизатора по очереди (одно изменение за раз) измените `weight_decay` (например, $0$, $10^{-4}$) и `label_smoothing`. Сведите результаты в таблицу.

Параметры `weight_decay` и `label_smoothing` должны быть явными аргументами ваших функций обучения, а не зашитыми константами.

In [ ]:
lr_choices = {}
optimizer_names = []
regularization_names = []
recipe3 = None
if schedule_recipe is not None:
    for opt_name, grid in [("sgd", [0.03, 0.1, 0.3]), ("adamw", [3e-4, 1e-3, 3e-3])]:
        candidates = []
        attempted_grid = 0
        for lr in grid:
            c = replace(schedule_recipe, optimizer=opt_name, lr=lr)
            name = f"grid_{opt_name}_{lr:g}"
            rows = group(name, c, "3", seeds=SEEDS[:1])
            if rows:
                attempted_grid += 1
            if rows and rows[0]["status"] == "complete":
                candidates.append((rows[0]["val_acc"], c))
        if attempted_grid == len(grid) and candidates:
            chosen = max(candidates, key=lambda x: x[0])[1]
            lr_choices[opt_name] = chosen
            name = f"optimizer_{opt_name}"
            group(name, chosen, "3")
            optimizer_names.append(name)
    if len(optimizer_names) == 2:
        opt_recipe = best_config(optimizer_names)
        if opt_recipe is not None:
            # Полная сетка: соседние пары отличаются ровно одним параметром.
            for wd in [0.0, 1e-4]:
                for smoothing in [0.0, 0.05]:
                    name = f"reg_wd{wd:g}_ls{smoothing:g}"
                    group(
                        name,
                        replace(opt_recipe, weight_decay=wd, label_smoothing=smoothing),
                        "3",
                    )
                    regularization_names.append(name)
            recipe3 = best_config(regularization_names)
grid_rows = []
for name, rows in RESULTS.items():
    if name.startswith("grid_"):
        for row in rows:
            grid_rows.append(
                dict(
                    name=name,
                    seed=row["seed"],
                    status=row["status"],
                    val_acc=row.get("val_acc"),
                    lr=row["config"]["lr"],
                )
            )
if grid_rows:
    display(pd.DataFrame(grid_rows))
summary(optimizer_names + regularization_names)
if len(optimizer_names) == 2:
    compare(*optimizer_names)
for a, b in [
    ("reg_wd0_ls0", "reg_wd0.0001_ls0"),
    ("reg_wd0_ls0", "reg_wd0_ls0.05"),
    ("reg_wd0.0001_ls0", "reg_wd0.0001_ls0.05"),
]:
    compare(a, b)
if recipe3 is None:
    print("Рецепт раздела 3 ещё не выбран: нужны результаты всех сравнений.")
else:
    display(pd.Series(asdict(recipe3), name="Выбранный рецепт"))


Ответьте на вопросы к заданию 3.3 (со ссылками на ваши числа):

1. Зачем нужен разогрев (warmup) и почему он особенно важен для `AdamW`?
2. Чем `weight_decay` в `AdamW` отличается от L2-регуляризации в `SGD`?
3. Как изменились val accuracy и val loss при `label_smoothing`? Можно ли сравнивать значения loss между запусками с разным `label_smoothing`?
4. Какой оптимизатор оказался лучше, и насколько надежен этот вывод?

### Ответы к 3.3

Warmup ограничивает первые обновления, пока веса и оценки градиентов ещё нестабильны. У AdamW в начале мало наблюдений для оценки моментов градиента, поэтому большой шаг может мешать обучению. Но преимущество warmup в этой работе можно установить только отдельным сравнением с запуском без него; текущие таблицы сравнивают целые расписания.

В SGD добавка `weight_decay * w` попадает в градиент, а при momentum также в накопленный момент. AdamW уменьшает веса отдельно от адаптивного пересчёта градиента. Поэтому одинаковое число `weight_decay` не означает одинаковую силу воздействия у двух оптимизаторов.

`label_smoothing` переносит часть массы правильного класса на остальные классы. Это снижает давление в сторону уверенных предсказаний. Значения обучающей loss с разным сглаживанием напрямую сравнивать нельзя: меняется целевая функция. В наших таблицах `val_loss` всегда считается без сглаживания, поэтому её сравнение корректно.

Численный ответ о победившем оптимизаторе и изменениях качества печатается в предыдущей ячейке. До обучения оснований выбирать SGD или AdamW по качеству нет.


## 4. Архитектурные эксперименты (2.5 балла)

В этом разделе мы попробуем улучшить базовую архитектуру.

__Общие условия.__ Чтобы различия определялись только архитектурой, все модели обучаем по одному рецепту с лучшим оптимизатором из предыдущего задания с фиксированнами гиперпараметрами, 20 эпох, без дополнительных аугментаций, если не сказано иное.

**Задание 4.1 (0.5 балла). Абляция skip-связи и BatchNorm.**

Сравните четыре варианта базовой сети из раздела 1: (a) полный, (b) без skip-связи, (c) без `BatchNorm2d`, (d) без skip-связи и без `BatchNorm2d`. Варианты должны отличаться только тем, что убрано (одинаковые инициализация, число каналов, сиды).

Для каждого варианта постройте графики train/val loss и accuracy (как в задании 2.2) и сведите val accuracy (по 3 запускам) в таблицу. Для вариантов без BatchNorm можно дополнительно сделать запуск с более подходящим lr. Если модель расходится при фиксированных параметрах оптимизатора, это результат эксперимента, а не ошибка.

In [ ]:
architecture_names = ["arch_basic", "arch_no_skip", "arch_no_bn", "arch_neither"]
if recipe3 is not None:
    for model_name, name in zip(
        ["basic", "no_skip", "no_bn", "neither"], architecture_names
    ):
        rows = group(
            name,
            replace(
                recipe3, model=model_name, epochs=20, augmentation="none", mixing="none"
            ),
            "4",
        )
        plot_history(rows, name)
summary(architecture_names)


Вопросы:

1. Почему BatchNorm позволяет использовать большую длину шага? Опирайтесь на ваши запуски.
2. Различаются ли варианты с skip-связью и без нее больше, чем на один std? Как вы думаете, почему?
3. Чем отличается поведение train loss и val loss в вариантах (a) и (d)?

BatchNorm нормирует активации по батчу и пространственным позициям. Масштабы внутри сети становятся устойчивее к изменению весов, поэтому иногда удаётся использовать больший шаг. Это не гарантия: результат при выбранном lr виден по кривым и случаям расходимости.

Skip-ветка даёт градиенту более короткий путь и сохраняет преобразованный вход. В такой короткой сети её эффект может быть небольшим. Размер эффекта и сравнение с одним std выводятся ниже.

Рост train loss и val loss вместе говорит о трудностях оптимизации. Падение train loss при росте val loss скорее указывает на переобучение. Сами по себе эти объяснения не доказывают, какое поведение получится у наших четырёх вариантов.


In [ ]:
compare("arch_basic", "arch_no_skip")
compare("arch_basic", "arch_neither")


**Задание 4.2 (0.5 балла). Параметры, FLOPs, рецептивное поле.**

1. __Параметры.__ Для каждого варианта из 4.1 выведите число параметров по слоям (покажите вычисление в markdown; не забудьте про параметры `BatchNorm2d`, bias и то, какие из них обучаемые), а затем проверьте результат в коде.
2. __FLOPs.__ Оцените число операций за один прямой проход для одного изображения $3 \times 32 \times 32$: сначала вручную для сверток и линейного слоя (выберите конвенцию — MACs или FLOPs $= 2 \cdot$ MACs — и явно укажите ее), затем проверьте результат в коде. Объясните расхождения, если они есть.
3. __Рецептивное поле.__ Вычислите рецептивное поле одного элемента выхода последнего сверточного слоя базовой сети. Можете использовать рекуррентные формулы: для слоя $l$ с ядром $k_l$ и шагом $s_l$ 
$$r_l = r_{l-1} + (k_l - 1)\, j_{l-1}, \qquad j_l = j_{l-1}\, s_l, \qquad r_0 = j_0 = 1.$$
Затем проверьте ответ экспериментально: например, по тому, на какие пиксели входа зависит выбранный элемент выхода (какая область входа имеет ненулевой градиент). Обоснуйте, почему проверка корректна. Почему эффективное рецептивное поле обычно меньше теоретического?

### Ручной подсчёт

Считаю MAC: одно умножение с накоплением. При конвенции «умножение и сложение отдельно» FLOPs = 2×MAC. В оценку MAC входят свёртки и линейные слои; BN, активации, сложения веток и pooling здесь не учтены.

Для свёртки число весов равно $C_{out}(C_{in}/g)k^2$, bias добавляет $C_{out}$ параметров. Число MAC равно $H_{out}W_{out}C_{out}(C_{in}/g)k^2$. Для linear: $d_{in}d_{out}+d_{out}$ параметров и $d_{in}d_{out}$ MAC. У BN обучаемы gamma и beta: $2C$. Среднее, дисперсия и счётчик батчей хранятся как буферы, в число обучаемых параметров не входят.

| Слой базовой сети | Параметры | MAC |
|---|---:|---:|
| conv1, 3→32, 3×3 | 32×3×9 + 32 = 896 | 32×32×32×3×9 = 884 736 |
| bn1 | 2×32 = 64 | вне выбранной оценки |
| conv2, 32→32, 3×3 | 32×32×9 + 32 = 9 248 | 32×32×32×32×9 = 9 437 184 |
| bn2 | 64 | вне выбранной оценки |
| shortcut, 3→32, 1×1 | 32×3 + 32 = 128 | 32×32×32×3 = 98 304 |
| linear, 512→10 | 512×10 + 10 = 5 130 | 5 120 |
| Всего | **15 530** | **10 425 344** |

Без skip вычитаются 128 параметров и 98 304 MAC. Без двух BN вычитаются 128 параметров, MAC в нашей конвенции не меняются. Если убрать оба компонента, остаётся 15 274 параметра и 10 327 040 MAC.

Для последней свёртки основной ветки $r_1=3$, $r_2=5$, шаг по входу $j=1$. Значит, теоретическое поле равно 5×5; у shortcut оно 1×1. Это поле одной позиции до пулинга. Итоговый классификатор собирает информацию со всех позиций, поэтому его зависимость от изображения шире.


In [ ]:
def parameter_table(model):
    rows = []
    for name, layer in model.named_modules():
        direct = list(layer.parameters(recurse=False))
        if direct:
            rows.append(
                dict(
                    layer=name,
                    total=sum(p.numel() for p in direct),
                    trainable=sum(p.numel() for p in direct if p.requires_grad),
                    buffers=sum(b.numel() for b in layer.buffers(recurse=False)),
                )
            )
    return pd.DataFrame(rows)


def profile_macs(model, shape=(1, 3, 32, 32)):
    model = copy.deepcopy(model).cpu().eval()
    rows = []
    hooks = []

    def hook(name):
        def collect(layer, inputs, output):
            if isinstance(layer, nn.Conv2d):
                mac = (
                    output.numel()
                    * (layer.in_channels // layer.groups)
                    * math.prod(layer.kernel_size)
                )
            else:
                mac = output.numel() * layer.in_features
            rows.append(dict(layer=name, macs=int(mac), shape=tuple(output.shape)))

        return collect

    try:
        for name, layer in model.named_modules():
            if isinstance(layer, (nn.Conv2d, nn.Linear)):
                hooks.append(layer.register_forward_hook(hook(name)))
        with torch.no_grad():
            model(torch.zeros(shape))
    finally:
        for h in hooks:
            h.remove()
    return pd.DataFrame(rows)


def model_cost(model):
    return sum(p.numel() for p in model.parameters()), int(
        profile_macs(model)["macs"].sum()
    )


def positive_probe(model):
    # Убираем случайное зануление градиентов ReLU; BN работает как поточечное преобразование.
    probe = copy.deepcopy(model).cpu().eval()
    with torch.no_grad():
        for layer in probe.modules():
            if isinstance(layer, nn.Conv2d):
                layer.weight.fill_(0.01)
                if layer.bias is not None:
                    layer.bias.fill_(0.1)
            elif isinstance(layer, nn.BatchNorm2d):
                layer.weight.fill_(1)
                layer.bias.zero_()
                layer.running_mean.zero_()
                layer.running_var.fill_(1)
    return probe


def receptive_field_check():
    probe = positive_probe(BasicBlockNet())
    x = torch.ones(1, 3, 32, 32, requires_grad=True)
    feature = probe.features(x)[0, 0, 16, 16]
    gradient = torch.autograd.grad(feature, x)[0].abs().sum((0, 1))
    yy, xx = torch.where(gradient > 0)
    observed = (int(yy.max() - yy.min() + 1), int(xx.max() - xx.min() + 1))
    assert observed == (5, 5), observed
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(gradient, cmap="magma")
    ax.set(title=f"Поддержка градиента: {observed}", xlabel="x входа", ylabel="y входа")
    plt.show()
    return observed


for kind in ["basic", "no_skip", "no_bn", "neither"]:
    m = build_model(Config(model=kind))
    display(Markdown(f"#### {kind}"))
    display(parameter_table(m))
    display(profile_macs(m))
    params, macs = model_cost(m)
    expected_params = (
        15530
        - (128 if kind in {"no_skip", "neither"} else 0)
        - (128 if kind in {"no_bn", "neither"} else 0)
    )
    expected_macs = 10425344 - (98304 if kind in {"no_skip", "neither"} else 0)
    assert params == expected_params and macs == expected_macs
receptive_field_check()


Вопросы:

1. Какая часть сети содержит больше всего параметров, а какая — больше всего FLOPs?
2. Покрывает ли рецептивное поле базовой сети все изображение $32 \times 32$? Как вы думаете, как это влияет на качество классификации?

Больше всего параметров и MAC у второй свёртки: 9 248 параметров и 9 437 184 MAC. У linear тоже заметная доля параметров (5 130), но он применяется один раз к вектору, а свёртка работает в каждой пространственной позиции.

Поле одной позиции 5×5 не покрывает всё изображение 32×32. Базовый блок видит местные признаки, а связывать удалённые детали ему трудно. При этом pooling и linear используют признаки со всего изображения, так что утверждение «классификатор видит только 5×5» было бы неверным.

Проверка градиентом выполнена в `eval`: BN не связывает разные пространственные позиции через статистики текущего батча. Положительные веса и вход не дают ReLU случайно скрыть возможные пути зависимости. Так проверяется геометрия архитектуры. У обученной модели карта влияния зависит и от весов. У обученной сети вклад дальних пикселей может быть слабым или нулевым; поэтому эффективное поле часто меньше теоретического.


**Задание 4.3 (0.75 балла). Downsampling-блок и малый ResNet.**

Блок из раздела 1 не меняет размер изображения, поэтому его нельзя использовать для постепенного уменьшения разрешения и увеличения числа каналов, как это делается в настоящих ResNet. Для этого нужен блок с downsampling:

- первая свертка $3 \times 3$ имеет `stride=2` и увеличивает число каналов в 2 раза, вторая свертка $3 \times 3$ сохраняет размер;
- после каждой свертки — `BatchNorm2d`, ReLU после сложения с shortcut;
- skip-связь не может быть тождественной (меняются и размер, и число каналов), поэтому в ней используется «проекция» — свертка $1 \times 1$ с тем же `stride=2` и `BatchNorm2d`.

Затем соберите малый ResNet в стиле CIFAR:

- stem: свертка $3 \times 3$ с $C$ каналами, `BatchNorm2d`, ReLU, без пулинга;
- стадия 1: один basic-блок ($C$ каналов);
- стадии 2 и 3: downsampling-блок и один basic-блок, число каналов удваивается на каждой стадии;
- global average pooling и линейный слой.

Число $C$ (например, 32 или 64) выберите сами.

Требования:

1. Проверьте формы выходов блока и сети с помощью `assert` (вход `(N, C, H, W)` → выход `(N, 2C, H/2, W/2)` для downsampling-блока).
2. Посчитайте параметры и FLOPs сети тем же способом, что и в 4.2, и обучите ее по рецепту раздела.
3. Добавьте результаты в сводную таблицу.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = (
            nn.Identity()
            if stride == 1 and in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(out)) + self.shortcut(x))


class SmallResNet(nn.Module):
    def __init__(self, C=32, blocks=(1, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, C, 3, padding=1, bias=False), nn.BatchNorm2d(C), nn.ReLU()
        )
        stages = []
        current = C
        for i, count in enumerate(blocks):
            out = C * 2**i
            stage = [ResidualBlock(current, out, stride=1 if i == 0 else 2)]
            stage.extend(ResidualBlock(out, out) for _ in range(count - 1))
            stages.append(nn.Sequential(*stage))
            current = out
        self.stages = nn.Sequential(*stages)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(4 * C, n_classes)

    def features(self, x):
        return self.stages(self.stem(x))

    def forward(self, x):
        return self.fc(self.pool(self.features(x)).flatten(1))


def resnet_rf(blocks=(1, 2, 2)):
    r, j = 3, 1  # stem
    for stage, count in enumerate(blocks):
        for b in range(count):
            stride = 2 if stage > 0 and b == 0 else 1
            r += 2 * j
            j *= stride
            r += 2 * j
    return r


with torch.no_grad():
    assert ResidualBlock(16, 32, 2)(torch.zeros(2, 16, 32, 32)).shape == (2, 32, 16, 16)
    assert SmallResNet()(torch.zeros(2, 3, 32, 32)).shape == (2, 10)
resnet_model = SmallResNet(32)
resnet_params, resnet_macs = model_cost(resnet_model)
display(parameter_table(resnet_model))
display(profile_macs(resnet_model))
print("ResNet:", resnet_params, "параметров;", resnet_macs, "MAC; поле", resnet_rf())
if recipe3 is not None:
    group(
        "arch_resnet",
        replace(
            recipe3,
            model="resnet",
            width=32,
            epochs=20,
            augmentation="none",
            mixing="none",
        ),
        "4",
    )


Вопросы:

1. Зачем в shortcut нужна свертка $1 \times 1$, и почему у нее такой же `stride`, как у основной ветки? Что произойдет, если взять `stride=1`?
2. Как выросли рецептивное поле, число параметров и FLOPs по сравнению с сетью из раздела 1, и как это отразилось на качестве и времени эпохи?
3. На какой стадии сосредоточено больше всего параметров, а на какой — больше всего FLOPs? Почему они совпадают / не совпадают?

Свёртка 1×1 в shortcut меняет число каналов. `stride=2` одновременно уменьшает высоту и ширину, чтобы обе ветки имели одинаковую форму. При `stride=1` в shortcut пространственные размеры не совпадут и сложение выдаст ошибку.

На длиннейшем пути малого ResNet поле последней свёртки равно 49×49: stem даёт 3, стадия 1 — 7, стадия 2 — 21, стадия 3 — 49. Оно уже покрывает вход 32×32, хотя на границах часть поля приходится на padding. Число параметров, MAC и время эпохи выводятся в таблицах; вывод о цене улучшения можно сделать после обучения.

Для каждого слоя ручной расчёт использует те же формулы, что в 4.2. Например, первая свёртка downsampling при входе H×H×C имеет `(2C)×C×9` весов и `(H/2)²×(2C)×C×9` MAC; её shortcut — `(2C)×C` весов и `(H/2)²×(2C)×C` MAC. В коде ниже эти выражения развёрнуты по стадиям и проверены независимыми хуками. В поздних стадиях больше параметров из-за широких каналов; выигрыш по площади изображения сдерживает рост MAC.


**Задание 4.4 (0.75 балла). ConvNeXt-подобный блок.**

После ResNet архитектуры сверточных сетей во многом «подтянулись» к трансформерам. Один из примеров — семейство [ConvNeXt](https://arxiv.org/abs/2201.03545). Реализуйте блок в его стиле:

- depthwise-свертка $7 \times 7$ (число групп равно числу каналов, размер изображения не меняется);
- `LayerNorm` по канальной оси (учтите, что в формате NCHW для этого нужна перестановка осей или специальная реализация);
- 1x1 (pointwise) расширение числа каналов в 4 раза;
- активация `GELU`;
- 1x1 сжатие обратно до исходного числа каналов;
- skip-connection (layer scale и stochastic depth реализовывать не нужно).

Соберите сеть с той же макроструктурой, что и в задании 4.3 (stem, три стадии, global average pooling, линейный слой). Между стадиями используйте downsampling в стиле ConvNeXt: `LayerNorm` и свертка $2 \times 2$ со stride 2. Число параметров сети должно отличаться от сети из 4.3 не более чем на 30%. Оптимизатор и `lr` подберите по val (такие сети обычно обучают `AdamW`).

In [ ]:
class ChannelLayerNorm(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        return self.norm(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)


class ConvNeXtBlock(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.dw = nn.Conv2d(C, C, 7, padding=3, groups=C)
        self.norm = ChannelLayerNorm(C)
        self.expand = nn.Conv2d(C, 4 * C, 1)
        self.compress = nn.Conv2d(4 * C, C, 1)

    def forward(self, x):
        return x + self.compress(F.gelu(self.expand(self.norm(self.dw(x)))))


class SmallConvNeXt(nn.Module):
    def __init__(self, C=48):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, C, 3, padding=1), ChannelLayerNorm(C))
        self.stages = nn.Sequential(
            nn.Sequential(ConvNeXtBlock(C)),
            nn.Sequential(
                ChannelLayerNorm(C),
                nn.Conv2d(C, 2 * C, 2, stride=2),
                ConvNeXtBlock(2 * C),
                ConvNeXtBlock(2 * C),
            ),
            nn.Sequential(
                ChannelLayerNorm(2 * C),
                nn.Conv2d(2 * C, 4 * C, 2, stride=2),
                ConvNeXtBlock(4 * C),
                ConvNeXtBlock(4 * C),
            ),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(4 * C, n_classes)

    def features(self, x):
        return self.stages(self.stem(x))

    def forward(self, x):
        return self.fc(self.pool(self.features(x)).flatten(1))


def convnext_rf():
    r, j = 3, 1
    r += 6 * j
    for _ in range(2):
        r += j
        j *= 2  # downsample 2×2
        r += 2 * 6 * j
    return r


# Подбор ширины только по числу параметров, без обучения и без val/test.
resnet_param_count = sum(p.numel() for p in SmallResNet(32).parameters())
CONVNEXT_WIDTH = min(
    range(24, 65, 4),
    key=lambda c: abs(
        sum(p.numel() for p in SmallConvNeXt(c).parameters()) - resnet_param_count
    ),
)
convnext_model = SmallConvNeXt(CONVNEXT_WIDTH)
cn_params, cn_macs = model_cost(convnext_model)
assert abs(cn_params / resnet_param_count - 1) <= 0.30
with torch.no_grad():
    assert convnext_model(torch.zeros(2, 3, 32, 32)).shape == (2, 10)
print(
    "ConvNeXt width:",
    CONVNEXT_WIDTH,
    "parameters:",
    cn_params,
    "MAC:",
    cn_macs,
    "ratio:",
    cn_params / resnet_param_count,
    "RF:",
    convnext_rf(),
)
convnext_names = []
if recipe3 is not None:
    for opt, lr in [("sgd", 0.03), ("adamw", 3e-4), ("adamw", 1e-3), ("adamw", 3e-3)]:
        name = f"cn_grid_{opt}_{lr:g}"
        c = replace(
            recipe3, model="convnext", width=CONVNEXT_WIDTH, optimizer=opt, lr=lr
        )
        group(name, c, "4", seeds=SEEDS[:1])
        convnext_names.append(name)
    display(
        pd.DataFrame(
            [
                dict(
                    name=n,
                    status=r["status"],
                    val_acc=r.get("val_acc"),
                    lr=r["config"]["lr"],
                )
                for n in convnext_names
                for r in RESULTS[n]
            ]
        )
    )
    candidates = [
        RESULTS[n][0]
        for n in convnext_names
        if RESULTS[n] and RESULTS[n][0]["status"] == "complete"
    ]
    if all(len(RESULTS[n]) == 1 for n in convnext_names) and candidates:
        c = Config(**max(candidates, key=lambda r: r["val_acc"])["config"])
        group("arch_convnext", c, "4")


Для блока с C каналами: depthwise 7×7 содержит `49C+C` параметров, LayerNorm — `2C`, расширение — `4C²+4C`, сжатие — `4C²+C`. Итого `8C²+57C`. MAC блока на карте H×H равны `H²(49C+8C²)`. Для downsampling C→2C: `2C` параметров LayerNorm и `4C(2C)+2C` параметров свёртки 2×2. Она работает на карте H/2×H/2. Ниже формулы проверяются по слоям независимо от хуков. Поле ConvNeXt по пространственной ветке: 3 → 9 → 10 → 34 → 36 → 84; увеличение после downsampling учитывает удвоившийся шаг.

In [ ]:
def manual_convnext_cost(C):
    rows = [dict(layer="stem", params=27 * C + C + 2 * C, macs=32 * 32 * 27 * C)]
    current = C
    h = 32
    for stage, count in enumerate([1, 2, 2]):
        channels = C * 2**stage
        if stage:
            h //= 2
            rows.append(
                dict(
                    layer=f"stage{stage+1}.down",
                    params=2 * current + 4 * current * channels + channels,
                    macs=h * h * 4 * current * channels,
                )
            )
        for b in range(count):
            rows.append(
                dict(
                    layer=f"stage{stage+1}.block{b+1}",
                    params=8 * channels**2 + 57 * channels,
                    macs=h * h * (49 * channels + 8 * channels**2),
                )
            )
        current = channels
    rows.append(dict(layer="fc", params=current * 10 + 10, macs=current * 10))
    return pd.DataFrame(rows)


cn_manual = manual_convnext_cost(CONVNEXT_WIDTH)
assert int(cn_manual.params.sum()) == cn_params
assert int(cn_manual.macs.sum()) == cn_macs
display(cn_manual)


Вопросы:

1. Назовите не менее четырех отличий этого блока от basic-блока ResNet и объясните, какую роль играет каждое.
2. Сопоставьте компоненты блока с блоком трансформера из семинара про ViT: что здесь соответствует перемешиванию информации между «токенами» (пространственными позициями), а что — перемешиванию каналов? Где в трансформере встречаются расширение в 4 раза, `GELU` и `LayerNorm`?
3. Почему в этом блоке используется `LayerNorm`, а не `BatchNorm`? Что изменится при очень малом размере батча?
4. Сравните сети из 4.3 и 4.4 по числу параметров, FLOPs, времени эпохи и качеству. Какая из них эффективнее и почему (с учетом того, что depthwise-свертки плохо загружают GPU)?

В этом блоке 7×7 depthwise-свёртка обрабатывает каждый канал отдельно. Она перемешивает соседние пространственные позиции с меньшим числом весов, чем обычная 7×7 свёртка. Свёртки 1×1 смешивают каналы, сначала расширяя представление в четыре раза. GELU меняет форму нелинейности по сравнению с ReLU. LayerNorm нормирует каналы отдельно в каждой позиции, без статистик батча. После сложения skip-ветки дополнительной активации нет.

В сравнении с ViT depthwise-свёртка играет роль пространственного смешивания, а две pointwise-свёртки похожи на MLP внутри блока трансформера. В этом MLP тоже встречаются расширение в четыре раза и GELU; LayerNorm обычно стоит перед attention и MLP. Аналогия ограничена: свёртка локальна и имеет фиксированные веса, attention может связывать удалённые позиции с весами, зависящими от входа.

LayerNorm не зависит от числа изображений в батче. При малом батче BatchNorm получает более шумные статистики; на ранних картах CIFAR часть проблемы смягчает усреднение по пространству. Нельзя заранее считать LayerNorm победителем по accuracy.

Сравнение эффективности требует и MAC, и времени эпохи. Depthwise-операции имеют мало арифметики относительно обмена данными и иногда используют GPU хуже плотных свёрток. Поэтому меньший MAC не обязательно означает более быструю эпоху. Фактическое соотношение видно в следующей таблице.


Выведите сводную таблицу по всем архитектурам раздела как в примере. Напоминаем, что все значения — mean ± std по 3 запускам:

| Модель | Параметры | FLOPs (MACs) | Receptive field | val acc | Время эпохи |
|---|---|---|---|---|---|
| Базовая (skip + BN) | | | | | |
| Без skip | | | | | |
| Без BN | | | | | |
| Без skip и без BN | | | | | |
| Малый ResNet | | | | | |
| ConvNeXt-подобная сеть | | | | | |

In [ ]:
def manual_resnet_cost(C=32):
    rows = []

    def conv(name, ci, co, k, h):
        rows.append(
            dict(layer=name, params=ci * co * k * k, macs=h * h * ci * co * k * k)
        )

    def bn(name, c):
        rows.append(dict(layer=name, params=2 * c, macs=0))

    conv("stem", 3, C, 3, 32)
    bn("stem_bn", C)
    ci = C
    h = 32
    for stage, count in enumerate([1, 2, 2]):
        co = C * 2**stage
        for block in range(count):
            down = stage > 0 and block == 0
            if down:
                h //= 2
            prefix = f"stage{stage+1}.block{block+1}"
            conv(prefix + ".conv1", ci, co, 3, h)
            bn(prefix + ".bn1", co)
            conv(prefix + ".conv2", co, co, 3, h)
            bn(prefix + ".bn2", co)
            if down:
                conv(prefix + ".shortcut", ci, co, 1, h)
                bn(prefix + ".shortcut_bn", co)
            ci = co
    rows.append(dict(layer="fc", params=ci * 10 + 10, macs=ci * 10))
    return pd.DataFrame(rows)


manual = manual_resnet_cost()
assert manual["params"].sum() == resnet_params
assert manual["macs"].sum() == resnet_macs
display(manual)
stage_costs = (
    manual.assign(stage=manual.layer.str.split(".").str[0])
    .groupby("stage", sort=False)[["params", "macs"]]
    .sum()
)
display(stage_costs)
print(
    "Больше параметров:",
    stage_costs["params"].idxmax(),
    "; больше MAC:",
    stage_costs["macs"].idxmax(),
)

arch_specs = [
    ("arch_basic", "basic", 32, 5),
    ("arch_no_skip", "no_skip", 32, 5),
    ("arch_no_bn", "no_bn", 32, 5),
    ("arch_neither", "neither", 32, 5),
    ("arch_resnet", "resnet", 32, resnet_rf()),
    ("arch_convnext", "convnext", CONVNEXT_WIDTH, convnext_rf()),
]
arch_table = []
for name, kind, width, rf in arch_specs:
    params, macs = model_cost(build_model(Config(model=kind, width=width)))
    rows = RESULTS.get(name, [])
    arch_table.append(
        dict(
            model=name,
            parameters=params,
            MACs=macs,
            receptive_field=rf,
            val_acc=fmt(rows, percent=True),
            epoch_seconds=fmt(rows, "epoch_seconds"),
        )
    )
display(pd.DataFrame(arch_table))
compare("arch_resnet", "arch_convnext")


## 5. Аугментации данных (1.5 балла)

Еще одной стандартной техникой, применяющейся в глубинном обучении, а особенно часто в компьютерном зрении, являются аугментации данных. Суть аугментаций состоит в том, что мы можем некоторым синтетическим образом видоизменять объекты обучающей выборки, тем самым расширяя ее, а также делая итоговую модель более устойчивой к таким изменениям.

Простейшая аугментация, которую можно применить к картинкам — разворот картинки по горизонтальной оси. То есть при обучении модели с вероятностью $0.5$ мы будем разворачивать картинку из обучающей выборки.

In [ ]:
if train_loader is not None:
    images, labels = next(iter(train_loader))
    imshow(torchvision.utils.make_grid(images[:4]), "Исходные изображения")
    imshow(torchvision.utils.make_grid(images[:4].flip(-1)), "Отражение по горизонтали")


Наиболее удобным способом работы с аугментациями в PyTorch является их задание в списке <code>transforms</code>, который затем передается в загрузчик данных. Обучим нашу сеть, применяя горизонтальные повороты:

In [ ]:
flip_rows = []
if recipe3 is not None:
    assert 10 <= AUG_EPOCHS <= 20
    print(
        "Flip-only опыт выполняется ниже после замера загрузчика, вместе с остальными аугментациями."
    )


Посчитайте метрики на валидационной выборке:

In [ ]:
summary(["aug_flip"])


**Задание 5.1 (0.25 балла). Конвейер данных на `torchvision.transforms.v2`.**

Напишите конвейер преобразований на `torchvision.transforms.v2` и сделайте его «базовым»: `RandomCrop(32, padding=4)` и `RandomHorizontalFlip`.Затем измерьте пропускную способность загрузки данных и объясните, где узкое место:

- замерьте время прохода по `train_loader` без модели (изображений в секунду) для разных `num_workers`, с и без `pin_memory`, с и без `persistent_workers`;
- либо перенесите аугментации на GPU (батчевые преобразования применяются к тензору, уже лежащему на GPU) и сравните со вторым вариантом.

Сведите замеры в таблицу и укажите выбранную конфигурацию, которую вы будете использовать дальше.

In [ ]:
def benchmark_loader():
    if not (RUN_DATA and RUN_BENCHMARK):
        print("Замеры выключены. Нужны RUN_DATA и RUN_BENCHMARK.")
        return None
    records = []
    for workers in [0, 2, 4]:
        for pin in [False, True]:
            for persistent in ([False, True] if workers else [False]):
                tr, va, cl = get_cifar10_data(
                    BATCH_SIZE,
                    make_transform("base"),
                    num_workers=workers,
                    pin_memory=pin,
                    persistent_workers=persistent,
                )
                # Отдельно первый проход с запуском workers и три тёплых прохода.
                times = []
                for repeat in range(4):
                    t = time.perf_counter()
                    count = 0
                    for x, y in tr:
                        count += len(y)
                    times.append(time.perf_counter() - t)
                rates = [count / t for t in times[1:]]
                records.append(
                    dict(
                        num_workers=workers,
                        pin_memory=pin,
                        persistent_workers=persistent,
                        first_seconds=times[0],
                        images_per_second=float(np.mean(rates)),
                        std=float(np.std(rates, ddof=1)),
                    )
                )
                del tr, va, cl
    frame = pd.DataFrame(records)
    display(frame)
    selected = frame.loc[frame.images_per_second.idxmax()]
    print("Самая быстрая загрузка без модели:", selected.to_dict())
    display(
        Markdown(
            "Этот замер включает чтение данных и CPU-преобразования, но не передачу "
            "на GPU. Поэтому он не устанавливает узкое место всего обучения. "
            "Сравнение num_workers показывает цену CPU-конвейера, а pin_memory "
            "нужно дополнительно оценивать по времени эпохи с моделью."
        )
    )
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    frame.to_csv(RESULTS_DIR / "loader-benchmark.csv", index=False)
    return frame


benchmark_path = RESULTS_DIR / "loader-benchmark.csv"
loader_benchmark = (
    pd.read_csv(benchmark_path)
    if benchmark_path.exists() and not RUN_BENCHMARK
    else benchmark_loader()
)
AUG_LOADER_OPTIONS = {}
if loader_benchmark is not None:
    selected_loader = loader_benchmark.loc[loader_benchmark.images_per_second.idxmax()]
    AUG_LOADER_OPTIONS = dict(
        num_workers=int(selected_loader.num_workers),
        pin_memory=bool(selected_loader.pin_memory),
        persistent_workers=bool(selected_loader.persistent_workers),
    )
# Меняем настройки только перед новыми опытами: они входят в ключ кэша.
if loader_benchmark is not None:
    winner = loader_benchmark.loc[loader_benchmark.images_per_second.idxmax()]
    print("Для новых аугментационных опытов используются:", AUG_LOADER_OPTIONS)


Для обучения дальше используется конфигурация с наибольшей скоростью загрузчика. Это предварительный выбор: быстрее читать батчи не всегда значит быстрее обучать сеть. Выбранные настройки сохраняются в `loader-benchmark.csv` и читаются при повторном запуске. Они явно входят в конфигурации новых аугментационных опытов. Настройки уже завершённых опытов разделов 2–4 не меняются. Число эпох и batch size для сравнения аугментаций остаются одинаковыми.


**Задание 5.2 (0.25 балла). `TrivialAugmentWide`, `RandAugment`, `RandomErasing`.**

Изучите эти преобразования в `torchvision.transforms.v2` и в своих словах объясните, как они устроены:

- `TrivialAugmentWide` и `RandAugment`: как выбираются операции и силы их применения, какие параметры есть у `RandAugment`;
- `RandomErasing`: что вырезается, какие параметры (`p`, `scale`, `ratio`) задают его силу, на каком этапе конвейера (до или после `Normalize`) он применяется и почему.

Для каждого преобразования визуализируйте сетку из нескольких аугментированных версий одного и того же изображения.

In [ ]:
def augmentation_gallery():
    if not RUN_DATA:
        print("Галереи появятся после включения RUN_DATA.")
        return
    raw = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
    image, label = raw[0]
    fig, axes = plt.subplots(3, 6, figsize=(12, 6))
    for row, name in enumerate(["trivial", "rand", "erase"]):
        transform = make_transform(name, erase_p=1.0)
        # Для галереи отключаем crop/flip, чтобы изолировать нужное преобразование.
        if name in {"trivial", "rand"}:
            op = (
                v2.TrivialAugmentWide()
                if name == "trivial"
                else v2.RandAugment(num_ops=2, magnitude=9)
            )
            transform = v2.Compose(
                [
                    v2.ToImage(),
                    op,
                    v2.ToDtype(torch.float32, scale=True),
                    v2.Normalize(CIFAR10_MEAN, CIFAR10_STD),
                ]
            )
        elif name == "erase":
            transform = v2.Compose(
                [
                    make_transform("none"),
                    v2.RandomErasing(
                        p=1.0, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0
                    ),
                ]
            )
        for col in range(6):
            axes[row, col].imshow(denormalize(transform(image)).permute(1, 2, 0))
            axes[row, col].axis("off")
            if col == 0:
                axes[row, col].set_title(name)
    fig.suptitle(f"Разные версии одного изображения: {classes[label]}")
    plt.tight_layout()
    plt.show()


augmentation_gallery()


`TrivialAugmentWide` выбирает одну операцию и случайную силу из доступного диапазона. В `RandAugment` задаются число операций `num_ops`, уровень силы `magnitude`, число уровней `num_magnitude_bins`; есть также настройки интерполяции и заполнения. Здесь используются две операции с силой 9 из 31 уровня.

`RandomErasing` заменяет прямоугольник константой. `p` задаёт вероятность применения, `scale` — долю площади, `ratio` — отношение сторон. Я применяю его после Normalize с `value=0`: заполнение соответствует среднему цвету датасета по каждому каналу. До Normalize нулевое заполнение было бы чёрным. В галерее `p=1`, чтобы стирание было видно; при обучении `p=0.25`.


**Задание 5.3 (0.5 балла). MixUp и CutMix.**

Эти методы изменяют не только изображения, но и метки. Для пары объектов $(x_i, y_i)$, $(x_j, y_j)$ и $\lambda \sim \mathrm{Beta}(\alpha, \alpha)$:

- MixUp: $\tilde x = \lambda x_i + (1 - \lambda) x_j$, $\tilde y = \lambda y_i + (1 - \lambda) y_j$;
- CutMix: в изображение $x_i$ вставляется прямоугольник из $x_j$, а $\lambda$ равно доле площади, оставшейся от $x_i$; метки смешиваются так же, как в MixUp.

Поскольку $\tilde y$ — вероятностное распределение по классам, для обучения нужна функция потерь с «мягкими» метками
$$\mathcal{L} = -\sum_{k} \tilde y_k \log p_k.$$

Требования:

1. Реализуйте такую функцию потерь самостоятельно и проверьте, что на one-hot метках она совпадает со стандартной кросс-энтропией для жестких меток.
2. Примените MixUp и CutMix к батчам обучающей выборки (можно использовать готовые преобразования из `torchvision.transforms.v2`, но формат и смысл меток вы должны объяснить).
3. Визуализируйте несколько смешанных изображений вместе с их мягкими метками.
4. Обучите модель с каждым из методов, сохраните результаты в таблицу на будущее.

In [ ]:
# Собственная soft_cross_entropy определена в разделе 2, потому что её использует train_epoch.
probe_logits = torch.tensor([[1.0, 2.0, -1.0], [0.0, -0.5, 0.7]], requires_grad=True)
probe_labels = torch.tensor([1, 2])
probe_onehot = F.one_hot(probe_labels, 3).float()
assert torch.allclose(
    soft_cross_entropy(probe_logits, probe_onehot),
    F.cross_entropy(probe_logits, probe_labels),
    atol=1e-7,
)
assert torch.allclose(
    torch.autograd.grad(soft_cross_entropy(probe_logits, probe_onehot), probe_logits)[
        0
    ],
    torch.autograd.grad(F.cross_entropy(probe_logits, probe_labels), probe_logits)[0],
)


def mixing_gallery():
    if not RUN_DATA:
        return
    loader, _, _ = get_cifar10_data(
        8, make_transform("base"), num_workers=0, pin_memory=False
    )
    x, y = next(iter(loader))
    fig, axes = plt.subplots(2, 4, figsize=(13, 6))
    for row, name in enumerate(["mixup", "cutmix"]):
        mixed, soft = make_mixer(name)(x.clone(), y.clone())
        assert soft.shape == (8, 10) and torch.allclose(soft.sum(1), torch.ones(8))
        for i in range(4):
            axes[row, i].imshow(denormalize(mixed[i]).permute(1, 2, 0))
            axes[row, i].axis("off")
            label = "; ".join(
                f"{classes[k]}: {float(p):.2f}" for k, p in enumerate(soft[i]) if p > 0
            )
            axes[row, i].set_title(f"{name}\n{label}", fontsize=8)
    plt.tight_layout()
    plt.show()


mixing_gallery()
if recipe3 is not None:
    for name in ["mixup", "cutmix"]:
        group(
            "aug_" + name,
            replace(
                recipe3,
                epochs=AUG_EPOCHS,
                augmentation="base",
                mixing=name,
                **AUG_LOADER_OPTIONS,
            ),
            "5",
        )


Вопросы:

1. Как корректно считать accuracy на обучающем батче при смешанных метках и стоит ли ее вообще интерпретировать?
2. Почему val loss и val accuracy считаются по жестким меткам и не используют смешивание?

После смешивания метки имеют форму `(batch, 10)`: каждая строка задаёт доли классов. У MixUp это коэффициент линейного смешивания, у CutMix — фактические доли площади после обрезания прямоугольника границами изображения.

В обучающем батче можно усреднить массу метки, приходящуюся на предсказанный класс. Именно это возвращает `batch_accuracy` для мягких меток. Но такое число не равно доле правильно классифицированных исходных картинок. Для таблицы и разрыва train−val я отдельно прогоняю чистую обучающую выборку без аугментаций в `eval` у модели последней эпохи. Этот прогон не меняет веса.

На val используются исходные изображения и жёсткие метки: нужно измерять решение исходной задачи. Смешивание сделало бы оценку другой задачей и добавило бы случайность.


**Задание 5.4 (0.5 балла). Таблица абляций.**

Постройте таблицу как в примере, где каждая строка — отдельная конфигурация аугментаций, обученная по лучшему рецепту из раздела 3 (3 запуска!):

| Конфигурация | train acc (последняя эпоха) | val acc | val loss | train−val gap |
|---|---|---|---|---|
| Только `RandomHorizontalFlip` | | | | |
| Базовая (crop + flip) | | | | |
| Базовая + `TrivialAugmentWide` | | | | |
| Базовая + `RandAugment` | | | | |
| Базовая + `RandomErasing` | | | | |
| Базовая + MixUp | | | | |
| Базовая + CutMix | | | | |
| Лучшая комбинация | | | | |

Если вычислительные мощности не позволяют, уменьшите число эпох одинаково для всех экпериментов (но не меньше 10) и укажите это.

In [ ]:
aug_names = [
    "aug_flip",
    "aug_base",
    "aug_trivial",
    "aug_rand",
    "aug_erase",
    "aug_mixup",
    "aug_cutmix",
]
if recipe3 is not None:
    group(
        "aug_flip",
        replace(
            recipe3,
            epochs=AUG_EPOCHS,
            augmentation="flip",
            mixing="none",
            **AUG_LOADER_OPTIONS
        ),
        "5",
    )
    for aug in ["base", "trivial", "rand", "erase"]:
        group(
            "aug_" + aug,
            replace(
                recipe3,
                epochs=AUG_EPOCHS,
                augmentation=aug,
                mixing="none",
                **AUG_LOADER_OPTIONS
            ),
            "5",
        )
    # Три комбинации выбираются только по val, без заранее назначенного победителя.
    combination_names = []
    for label, aug, mix in [
        ("rand_erase", "rand_erase", "none"),
        ("rand_mixup", "rand", "mixup"),
        ("erase_cutmix", "erase", "cutmix"),
    ]:
        name = "combo_" + label
        group(
            name,
            replace(
                recipe3,
                epochs=AUG_EPOCHS,
                augmentation=aug,
                mixing=mix,
                **AUG_LOADER_OPTIONS
            ),
            "5",
        )
        combination_names.append(name)
    best_combination = best_config(combination_names)
    if best_combination is not None:
        group("aug_best_combination", best_combination, "5")
summary(
    aug_names + ["aug_best_combination"],
    metrics=("train_acc", "last_val_acc", "last_val_loss", "gap", "val_acc"),
)
# train_acc / last_val_* / gap относятся к последней эпохе; val_acc — к выбранному checkpoint.


1. Какие аугментации дали улучшение, превышающее один std, и какие — нет?
2. Где по разрыву train−val видно переобучение, и как его меняют аугментации?
3. Почему сильные аугментации (например, MixUp, CutMix) могут ухудшать качество при коротком обучении в 20 эпох? Подтверждают ли это ваши результаты?

In [ ]:
for name in [
    "aug_trivial",
    "aug_rand",
    "aug_erase",
    "aug_mixup",
    "aug_cutmix",
    "aug_best_combination",
]:
    compare("aug_base", name)
for name in aug_names + ["aug_best_combination"]:
    rows = RESULTS.get(name, [])
    if complete(rows):
        gap, sd = stats(rows, "gap")
        print(f"{name}: train−val последней эпохи = {100*gap:.2f} ± {100*sd:.2f} п.п.")


Большой положительный разрыв train−val указывает на переобучение, но сам по себе не описывает качество модели: маленький разрыв бывает и у плохо обученной сети. Поэтому смотрю на него вместе с val accuracy.

Сильное смешивание усложняет задачу и может замедлить обучение. При 20 эпохах сеть иногда не успевает получить пользу от такой регуляризации. Проверить это объяснение можно по приведённым сравнениям MixUp/CutMix с `aug_base`; если разница меньше одного std, надёжного ухудшения по правилу задания нет. Без чисел такое объяснение остаётся гипотезой. Проверка влияния длительности потребовала бы отдельного опыта с большим числом эпох.


## 6. Анализ ошибок и интерпретируемость (1.25 балла)

В этом разделе вы проанализируете лучшую модель, выбранную по val из разделов 3–5.

**Задание 6.1 (0.5 балла). Per-class accuracy и confusion matrix.**

1. Посчитайте accuracy по каждому классу на test и постройте confusion matrix (при необходимости нормируйте по строкам).
2. Выведите три наиболее частые пары перепутанных классов.
3. Отдельно разберите пару cat/dog: сколько кошек классифицировано как собаки и наоборот, покажите несколько ошибочно классифицированных изображений этой пары и предложите объяснения.
4. Сравните per-class accuracy вашей лучшей модели с моделью из раздела 1: какие классы улучшились сильнее всего?

In [ ]:
selection_names = ["baseline", "multistep", "constant_0.001", "onecycle", "cosine"]
selection_names += optimizer_names + regularization_names
selection_names += architecture_names + ["arch_resnet", "arch_convnext"]
selection_names += aug_names + ["aug_best_combination"]
# Выбор идёт по среднему val трёх сидов; test ещё не создан.
available_names = [n for n in selection_names if complete(RESULTS.get(n, []))]
selection_ready = recipe3 is not None and all(
    attempted(RESULTS.get(n, []))
    for n in architecture_names
    + ["arch_resnet", "arch_convnext"]
    + aug_names
    + ["aug_best_combination"]
)
best_name = (
    max(available_names, key=lambda n: stats(RESULTS[n])[0])
    if selection_ready
    else None
)


def load_model(row, on_device=device):
    if row["status"] != "complete":
        raise ValueError("Нет завершённой модели.")
    model = build_model(Config(**row["config"]))
    state = torch.load(
        Path(row["path"]) / "best.pt", map_location="cpu", weights_only=True
    )
    model.load_state_dict(state)
    return model.to(on_device).eval()


@torch.no_grad()
def predictions(model, loader):
    true, pred = [], []
    model_device = next(model.parameters()).device
    for x, y in loader:
        pred.append(model(x.to(model_device, non_blocking=True)).argmax(1).cpu())
        true.append(y)
    return torch.cat(true).numpy(), torch.cat(pred).numpy()


def cached_predictions(row, loader):
    dest = Path(row["path"]) / "test-predictions.npz"
    if dest.exists():
        with np.load(dest) as saved:
            return saved["true"], saved["pred"]
    model = load_model(row)
    try:
        true, pred = predictions(model, loader)
        np.savez_compressed(dest, true=true, pred=pred)
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return true, pred


def per_class_accuracy(true, pred):
    cm = confusion_matrix(true, pred, labels=np.arange(10))
    return cm.diagonal() / cm.sum(1), cm


def show_error_images(dataset, indices, pred, title):
    if not len(indices):
        print(title, ": таких ошибок нет.")
        return
    indices = list(indices)[:6]
    fig, axes = plt.subplots(
        1, len(indices), figsize=(3 * len(indices), 3), squeeze=False
    )
    for ax, idx in zip(axes.flat, indices):
        x, y = dataset[int(idx)]
        ax.imshow(denormalize(x).permute(1, 2, 0))
        ax.axis("off")
        ax.set_title(f"#{idx}: {classes[y]} → {classes[pred[idx]]}", fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


test_loader = None
error_report = None
if RUN_REPORTS and "6" in STAGES:
    if best_name is None or not complete(baseline_rows):
        raise RuntimeError("Сначала нужны baseline и завершённые сравнения по val.")
    print("Выбрано по val:", best_name, "; val:", fmt(RESULTS[best_name], percent=True))
    test_loader = get_test_loader()
    best_class = []
    base_class = []
    best_cms = []
    for best_row, base_row in zip(RESULTS[best_name], baseline_rows):
        y, pred = cached_predictions(best_row, test_loader)
        y0, pred0 = cached_predictions(base_row, test_loader)
        a, cm = per_class_accuracy(y, pred)
        a0, _ = per_class_accuracy(y0, pred0)
        best_class.append(a)
        base_class.append(a0)
        best_cms.append(cm)
    best_class = np.array(best_class)
    base_class = np.array(base_class)
    per_class = pd.DataFrame(
        dict(
            class_name=classes,
            baseline_mean=base_class.mean(0),
            baseline_std=base_class.std(0, ddof=1),
            best_mean=best_class.mean(0),
            best_std=best_class.std(0, ddof=1),
            improvement=best_class.mean(0) - base_class.mean(0),
        )
    )
    display(per_class.sort_values("improvement", ascending=False))
    # Сумма матриц: каждое изображение оценено тремя независимо обученными моделями.
    cm_sum = np.sum(best_cms, axis=0)
    fig, ax = plt.subplots(figsize=(9, 8))
    ConfusionMatrixDisplay(
        cm_sum / cm_sum.sum(1, keepdims=True), display_labels=classes
    ).plot(ax=ax, xticks_rotation=45, values_format=".2f", colorbar=False)
    ax.set_title("Лучшая модель: доли по истинному классу, три сида")
    plt.show()
    pairs = sorted(
        [
            (int(cm_sum[i, j] + cm_sum[j, i]), classes[i], classes[j])
            for i in range(10)
            for j in range(i + 1, 10)
        ],
        reverse=True,
    )[:3]
    display(
        pd.DataFrame(pairs, columns=["ошибок за три запуска", "класс A", "класс B"])
    )
    print("Cat → dog по сидам:", [int(cm[3, 5]) for cm in best_cms])
    print("Dog → cat по сидам:", [int(cm[5, 3]) for cm in best_cms])
    # Для изображений и Grad-CAM берём первый заранее заданный seed, не лучший test seed.
    y, pred = cached_predictions(RESULTS[best_name][0], test_loader)
    pair_errors = np.flatnonzero(((y == 3) & (pred == 5)) | ((y == 5) & (pred == 3)))
    show_error_images(
        test_loader.dataset, pair_errors, pred, "Ошибки cat/dog, первый seed"
    )
    error_report = dict(
        true=y, pred=pred, pair_errors=pair_errors, cm=cm_sum, per_class=per_class
    )
else:
    print("Test не читался: RUN_REPORTS=False или раздел 6 не включён.")


У кошек и собак бывают похожие позы, шерсть и фон; в 32×32 мелкие детали теряются. Это возможное объяснение ошибок. По одной confusion matrix нельзя установить причину. Сначала нужно посмотреть конкретные картинки и карты Grad-CAM. Числа cat→dog и dog→cat выводятся отдельно для каждого сида, а три пары классов ранжируются по сумме ошибок в обоих направлениях.


**Задание 6.2 (0.75 балла). Grad-CAM.**

[Grad-CAM](https://arxiv.org/abs/1610.02391) показывает, какие области изображения повлияли на предсказание класса $c$. Для выбранного сверточного слоя с картами признаков $A^k$ и логита $y^c$ веса каналов равны усредненным по пространству градиентам, $\alpha_k^c = \frac{1}{HW} \sum_{i,j} \frac{\partial y^c}{\partial A^k_{ij}}$, а карта важности — $L^c = \mathrm{ReLU}\bigl(\sum_k \alpha_k^c A^k\bigr)$, которую масштабируют до размера входного изображения и накладывают на него (с прозрачностью).

Требования:

1. Реализуйте Grad-CAM самостоятельно (например, через forward- и backward-хуки на выбранном слое). Использовать готовые библиотеки Grad-CAM нельзя.
2. Объясните, почему вы выбрали именно этот слой.
3. Покажите Grad-CAM не менее чем для 4 правильно и 4 неправильно классифицированных изображений test (среди ошибочных — хотя бы одна пара cat/dog), для предсказанного и истинного классов.
4. Проверьте, что карта действительно зависит от модели: сравните Grad-CAM для одного и того же изображения у модели из раздела 1 и у вашей лучшей модели.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model = model
        self.activation = None
        self.handle = layer.register_forward_hook(self._save_activation)

    def _save_activation(self, module, inputs, output):
        self.activation = output

    def __call__(self, image, class_index=None):
        self.model.eval()
        # enable_grad нужен, даже если вызывающий код случайно включил no_grad.
        with torch.enable_grad():
            x = (
                image.detach()
                .clone()
                .to(next(self.model.parameters()).device)
                .requires_grad_(True)
            )
            logits = self.model(x)
            cls = (
                int(logits.argmax(1).item())
                if class_index is None
                else int(class_index)
            )
            gradients = torch.autograd.grad(logits[0, cls], self.activation)[0]
            weights = gradients.mean(dim=(-2, -1), keepdim=True)
            cam = F.relu((weights * self.activation).sum(dim=1, keepdim=True))
            cam = F.interpolate(
                cam, size=x.shape[-2:], mode="bilinear", align_corners=False
            )
            cam = cam[0, 0].detach().cpu()
            cam = (cam - cam.min()) / (cam.max() - cam.min()).clamp_min(1e-12)
        self.activation = None
        return cam, cls

    def close(self):
        self.handle.remove()


def cam_layer(model):
    if isinstance(model, BasicBlockNet):
        return model.conv2
    if isinstance(model, SmallConvNeXt):
        return model.stages[-1][-1].dw
    return model.stages[-1][-1].conv2


def draw_cam(ax, image, cam, title):
    ax.imshow(denormalize(image.cpu()).permute(1, 2, 0))
    ax.imshow(cam, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    ax.set_title(title, fontsize=8)
    ax.axis("off")


def gradcam_report():
    if error_report is None:
        return
    true, pred = error_report["true"], error_report["pred"]
    correct = list(np.flatnonzero(true == pred)[:4])
    wrong = list(np.flatnonzero(true != pred))
    pair = list(error_report["pair_errors"])
    # В ошибочную четвёрку сначала включаем пример cat/dog, если он есть.
    chosen_wrong = (pair[:1] + [i for i in wrong if i not in pair[:1]])[:4]
    if len(correct) < 4 or len(chosen_wrong) < 4:
        raise RuntimeError(
            "В test недостаточно правильных или ошибочных примеров для задания."
        )
    if not pair:
        raise RuntimeError("Нет ошибок cat/dog: это нужно отдельно отметить в отчёте.")
    indices = correct + chosen_wrong
    model = load_model(RESULTS[best_name][0])
    cam = GradCAM(model, cam_layer(model))
    try:
        fig, axes = plt.subplots(8, 3, figsize=(10, 23))
        for row, idx in enumerate(indices):
            image, label = test_loader.dataset[int(idx)]
            pmap, pcls = cam(image[None], int(pred[idx]))
            tmap, _ = cam(image[None], int(label))
            axes[row, 0].imshow(denormalize(image).permute(1, 2, 0))
            axes[row, 0].axis("off")
            axes[row, 0].set_title(
                f"#{idx}, true={classes[label]}, pred={classes[pcls]}", fontsize=8
            )
            draw_cam(axes[row, 1], image, pmap, "Предсказанный класс: " + classes[pcls])
            draw_cam(axes[row, 2], image, tmap, "Истинный класс: " + classes[label])
        plt.tight_layout()
        plt.show()
    finally:
        cam.close()
        del model
    # Одно и то же изображение и один и тот же целевой класс у двух моделей.
    idx = chosen_wrong[0]
    image, label = test_loader.dataset[int(idx)]
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    for ax, row, name in zip(
        axes, [baseline_rows[0], RESULTS[best_name][0]], ["baseline", best_name]
    ):
        model = load_model(row)
        cam = GradCAM(model, cam_layer(model))
        try:
            heatmap, _ = cam(image[None], label)
            draw_cam(ax, image, heatmap, f"{name}: true={classes[label]}")
        finally:
            cam.close()
            del model
    plt.tight_layout()
    plt.show()


gradcam_report()


Вопросы к заданию 6.2:

1. На что смотрит модель при правильных и неправильных предсказаниях? Есть ли признаки того, что модель использует фон или контекст?
2. Какие вы видите минусы у такого способа визуализации?

Для базовой сети беру вторую свёртку. Для ResNet — последнюю свёртку последнего блока; для ConvNeXt — depthwise-свёртку последнего блока. В этих слоях сохраняются пространственные координаты и уже есть признаки поздней стадии. У более глубоких сетей карта грубее, потому что пространственное разрешение уменьшилось до 8×8.

Тёплые области показывают положительный вклад выбранных карт признаков в логит класса после усреднения градиентов. Они не являются точной разметкой объекта и не доказывают причинную связь. Одинаковые карты для истинного и предсказанного класса у правильного ответа ожидаемы: это один класс.

После запуска нужно описать показанные случаи: где карта попадает на животное, где выделяет фон, чем отличаются карты baseline и выбранной модели. Вывод «модель использует фон» допустим как наблюдение о конкретной карте; для проверки потребовалось бы, например, закрыть фон и повторить предсказание. До построения карт таких наблюдений нет.

Ограничения Grad-CAM: зависимость от выбранного слоя, потеря деталей при увеличении карты, усреднение разных пространственных градиентов, отбрасывание отрицательных вкладов через ReLU. Яркость после нормировки нельзя использовать для сравнения абсолютной силы влияния разных моделей. Если baseline и победитель совпадут, их карты совпадут тоже; это не независимая проверка зависимости от модели. В таком случае ниже используется другая обученная архитектура.


In [ ]:
if error_report is not None and best_name == "baseline":
    alternatives = [
        n
        for n in ["arch_resnet", "arch_convnext", "aug_base"]
        if complete(RESULTS.get(n, []))
        and RESULTS[n][0]["path"] != baseline_rows[0]["path"]
    ]
    if alternatives:
        idx = int(error_report["pair_errors"][0])
        image, label = test_loader.dataset[idx]
        fig, axes = plt.subplots(1, 2, figsize=(7, 3))
        for ax, row, name in zip(
            axes,
            [baseline_rows[0], RESULTS[alternatives[0]][0]],
            ["baseline", alternatives[0]],
        ):
            model = load_model(row)
            cam = GradCAM(model, cam_layer(model))
            try:
                heatmap, _ = cam(image[None], label)
                draw_cam(ax, image, heatmap, name)
            finally:
                cam.close()
                del model
        plt.tight_layout()
        plt.show()
    else:
        print("Для независимой проверки Grad-CAM пока нет второй обученной модели.")


## 7. Челлендж (1.5 балла)

Постройте лучшую модель, которую сможете, в рамках ограничений:

- не более 30 эпох;
- не более 10 млн параметров;
- обучение на одной GPU;
- только обучающая часть CIFAR-10 (без внешних данных и предобученных весов);
- модель и гиперпараметры выбираются по val, test используется один раз для итоговой оценки.

Вы сами выбираете архитектуру, оптимизатор, расписание, аугментации и регуляризацию, используя идеи из разделов 3–5 (и любые другие).

Что нужно сдать:

1. __Описание решения__ (0.25 балла): текст / таблицы со всеми компонентами (архитектура, число параметров, число эпох, размер батча, оптимизатор, `lr`-расписание, аугментации, регуляризация) и временем одной эпохи.
2. __Абляции__ (0.5 балла): удалите или замените по очереди не менее трех компонентов вашего рецепта и покажите, как меняется val accuracy (не забываем про 3 запуска для финального решения и для каждой абляции). Укажите хотя бы одну идею, которая не сработала.
3. __Обоснование__ (0.25 балла): для каждого компонента рецепта объясните, почему вы его включили, со ссылкой на числа из абляций.
4. __Итоговое качество__ (0.5 балла): test accuracy (mean ± std). Качество $\geqslant 90\%$ — полный балл; от 85% до 90% — 0.25; ниже 85% — 0 баллов.

### Рецепт для челленджа

Проверяю ResNet с двумя блоками на каждой из трёх стадий и шириной 64. У него есть путь для передачи градиента через skip-связи, а downsampling увеличивает поле зрения. В качестве начального кандидата беру SGD с momentum, cosine с трёхэпохным warmup, crop+flip, `weight_decay=5e-4` и `label_smoothing=0.05`. Качество этого рецепта ещё предстоит измерить.

Сравниваю три удаления по одному: без crop+flip, без weight decay, без label smoothing. Дополнительно проверяю RandAugment, который за короткое обучение может не помочь. В коде победитель выбирается только по среднему val; если победит изменённый рецепт, для него автоматически строится свой набор трёх абляций. Test вызывается только после этого выбора. «Не сработало» пишется лишь тогда, когда это следует из результатов.


In [ ]:
class ChallengeResNet(SmallResNet):
    def __init__(self, C=64):
        super().__init__(C, blocks=(2, 2, 2))


challenge_seed_config = Config(
    model="challenge",
    width=64,
    optimizer="sgd",
    lr=0.1,
    schedule="cosine",
    weight_decay=5e-4,
    label_smoothing=0.05,
    augmentation="base",
    epochs=30,
)
challenge_nparams = sum(
    p.numel() for p in build_model(challenge_seed_config).parameters()
)
assert challenge_nparams <= 10_000_000 and challenge_seed_config.epochs <= 30
print("Челлендж:", challenge_nparams, "параметров; 30 эпох; одна GPU; веса с нуля.")
challenge_candidates = {
    "challenge_initial": challenge_seed_config,
    "challenge_no_aug": replace(challenge_seed_config, augmentation="none"),
    "challenge_no_wd": replace(challenge_seed_config, weight_decay=0.0),
    "challenge_no_ls": replace(challenge_seed_config, label_smoothing=0.0),
    "challenge_rand": replace(challenge_seed_config, augmentation="rand"),
}
for name, c in challenge_candidates.items():
    group(name, c, "7")
summary(list(challenge_candidates))
challenge_recipe = best_config(list(challenge_candidates))
challenge_final_rows = []
challenge_ablation_names = []
if challenge_recipe is not None:
    challenge_final_rows = group("challenge_final", challenge_recipe, "7")
    modifications = [
        ("augmentation", "none" if challenge_recipe.augmentation != "none" else "base"),
        ("weight_decay", 0.0 if challenge_recipe.weight_decay else 5e-4),
        ("label_smoothing", 0.0 if challenge_recipe.label_smoothing else 0.05),
    ]
    for field, value in modifications:
        name = "challenge_ablate_" + field
        group(name, replace(challenge_recipe, **{field: value}), "7")
        challenge_ablation_names.append(name)
    display(pd.Series(asdict(challenge_recipe), name="Финальный рецепт, выбран по val"))
    summary(["challenge_final"] + challenge_ablation_names)
    for name in challenge_ablation_names:
        compare("challenge_final", name)
    print(
        "Среднее время обучающей эпохи, секунды:",
        fmt(challenge_final_rows, "epoch_seconds"),
    )
    # Обоснование не предполагает, что каждое добавление обязательно полезно.
    for field, name in zip(
        ["аугментации", "weight_decay", "label_smoothing"], challenge_ablation_names
    ):
        if complete(RESULTS[name]) and complete(challenge_final_rows):
            m, s = stats(challenge_final_rows)
            ma, sa = stats(RESULTS[name])
            print(
                f"{field}: финальный рецепт {100*m:.2f}±{100*s:.2f}%, "
                f"изменённый {100*ma:.2f}±{100*sa:.2f}%; "
                + (
                    "текущая настройка лучше по правилу одного std."
                    if m - ma > max(s, sa)
                    else "однозначного преимущества компонента по этому сравнению нет."
                )
            )
    failed = [
        n
        for n in challenge_candidates
        if complete(RESULTS[n])
        and stats(challenge_final_rows)[0] - stats(RESULTS[n])[0]
        > max(stats(challenge_final_rows)[1], stats(RESULTS[n])[1])
    ]
    if failed:
        print("Идеи, уступившие победителю больше чем на один std:", ", ".join(failed))
    else:
        losers = [
            n
            for n in challenge_candidates
            if complete(RESULTS[n])
            and RESULTS[n][0]["config"] != asdict(challenge_recipe)
        ]
        if losers:
            attempt = min(losers, key=lambda n: stats(RESULTS[n])[0])
            print(
                "Не повысил среднее val относительно выбранного рецепта:",
                attempt,
                "; val =",
                fmt(RESULTS[attempt], percent=True),
                ". Разница укладывается в один std: доказанного ухудшения нет.",
            )


In [ ]:
def final_challenge_test():
    if not FINAL_TEST:
        print("Финальный test выключен.")
        return
    if (
        len(challenge_ablation_names) != 3
        or not complete(challenge_final_rows)
        or not all(attempted(RESULTS.get(n, [])) for n in challenge_ablation_names)
    ):
        raise RuntimeError(
            "Сначала нужны финальный рецепт и три абляции по три запуска."
        )
    # Файл фиксирует рецепт: случайное повторное выполнение не оценивает другую модель.
    manifest = RESULTS_DIR / "challenge-final-test.json"
    signature = [
        experiment_path(Config(**r["config"]), r["seed"]).name
        for r in challenge_final_rows
    ]
    if manifest.exists():
        saved = json.loads(manifest.read_text())
        if saved["signature"] != signature:
            raise RuntimeError(
                "Test уже использовался для другого рецепта. Повторный подбор запрещён."
            )
    else:
        loader = get_test_loader()
        accuracies = []
        for row in challenge_final_rows:
            y, pred = cached_predictions(row, loader)
            accuracies.append(float((y == pred).mean()))
        saved = dict(
            signature=signature, config=asdict(challenge_recipe), test_acc=accuracies
        )
        manifest.write_text(json.dumps(saved, ensure_ascii=False, indent=2))
    values = np.array(saved["test_acc"])
    mean, std = values.mean(), values.std(ddof=1)
    print(f"Итоговая test accuracy: {100*mean:.2f} ± {100*std:.2f}%")
    print("Баллы за качество:", 0.5 if mean >= 0.90 else 0.25 if mean >= 0.85 else 0)
    print("Бонус за 94%:", int(mean >= 0.94))


final_challenge_test()


## Бонус. Логирование в comet (1 балл)

На практике специалиста по глубинному обучению часто встречаются ситуации, когда нейросеть учится на каком-то удаленном сервере. И обычно вам хочется отслеживать прогресс обучения, особенно когда время обучения модели исчисляется днями или неделями. Для таких целей существует несколько инструментов. Например: [TensorBoard](https://docs.pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html), [comet](https://www.comet.com), [wandb](https://wandb.ai/). Выберите любой удобный из них.

Ваша задача состоит в том, чтобы разобраться как им пользоваться, и повторить задания 2.1 и 2.2 с его использованием. Обучение вы можете запускать в этом же ноутбуке, но теперь вам необходимо логировать значения функции потерь и точности на обучающей выборке и на валидационной. Результатом работы должны быть ваш код, публичная ссылка на страничку с графиками, идентичными графикам в задании 2.2 или выхлоп тензорбоард.

### Бонус: TensorBoard

Выбран TensorBoard. При `USE_TENSORBOARD=True` функция `train` записывает train loss и accuracy по батчам, val loss и accuracy раз в эпоху, а также learning rate. Шаги на оси X совпадают с графиками раздела 2. Для смешанных меток обучающая accuracy подписана как `train_augmented`.

Чтобы повторить базовый опыт с логированием, включите `USE_TENSORBOARD` и раздел `"bonus"`. Логи сохраняются рядом с весами. В Colab откройте их командами `%load_ext tensorboard` и `%tensorboard --logdir /путь/к/hw2-results`. Для сдачи приложите вывод TensorBoard или экспорт графиков. Здесь бонус пока не подтверждён: логов обученного запуска ещё нет.


In [ ]:
if USE_TENSORBOARD and "bonus" in STAGES and RUN_TRAINING:
    # Своя папка, чтобы наличие обычного baseline не пропустило запись логов.
    old_results_dir = RESULTS_DIR
    try:
        RESULTS_DIR = old_results_dir / "baseline-with-tensorboard"
        bonus_rows = group("baseline_tensorboard", baseline_config, "bonus")
        plot_history(bonus_rows, "Baseline с TensorBoard")
    finally:
        RESULTS_DIR = old_results_dir


## Бонус. Качество 94%+ (1 балл)

Если средняя по 3 запускам test accuracy вашей модели не ниже 94% при ограничениях пункта 7, вы получаете 1 дополнительный балл. Бонус засчитывается только при наличии таблицы абляций и воспроизводимого кода.

### Источники и состояние работы

Использованы условия исходного ноутбука и документация:

- [Преобразования torchvision v2](https://docs.pytorch.org/vision/stable/transforms.html), [MixUp](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.v2.MixUp.html), [RandomErasing](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.v2.RandomErasing.html).
- [Расписания learning rate в PyTorch](https://docs.pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate).
- [ConvNeXt, Liu et al.](https://arxiv.org/abs/2201.03545): идея блока с depthwise-свёрткой, LayerNorm и расширением каналов.
- [Grad-CAM, Selvaraju et al.](https://arxiv.org/abs/1610.02391): формула карты и усреднения градиентов.

Код и пояснения подготовлены с помощью Codex; текст отредактирован по правилам humanizer. Готовые решения других студентов не использовались. Все Grad-CAM и функция потерь для мягких меток реализованы в этом ноутбуке.

В этой версии нет результатов обучения. До сдачи нужно выполнить эксперименты, сохранить выводы ячеек и проверить, что объяснения соответствуют полученным графикам. Если разница не превышает один std, код так и пишет. Достичь 90% или 94% заранее не обещается.
